# Siniestros de tránsito en Ecuador, enero a septiembre de 2026
## Proyecto 1P · Introducción a la Ciencia de Datos

**Grupo #9:** Bruno Romero, Darwin Diaz y David Sandoval

---

### Objetivo

Describir **cuándo, dónde y por qué** ocurrieron los siniestros de tránsito y **a quién afectaron**, e identificar qué variables explican mejor lo sucedido.

Es un análisis **exploratorio, descriptivo y retrospectivo**: caracteriza los datos ya registrados. No predice dónde ocurrirá el próximo siniestro ni usa modelos de aprendizaje automático.

### Datos

Base de la Agencia Nacional de Tránsito (ANT): **17 236 siniestros** y **374 columnas** (archivo `08. BDD_Septiembre_2026_SM.xlsx`). Cada fila es un siniestro y describe hasta 43 personas y 13 vehículos. Incluye datos temporales (fecha y hora), categóricos (tipo y causa), continuos (edad, número de víctimas) y espaciales (provincia, cantón, latitud y longitud).

### Contenido

| Bloque | Pregunta | Tema |
|---|---|---|
| Cuándo | 1 | Meses, días y horas con más siniestros; urbano vs rural |
| Cuándo | 2 | Días críticos y qué tienen en común |
| Dónde | 3 | Provincias y cantones que concentran siniestros y muertes |
| Dónde | 4 | Provincias donde pesan más motos, camiones y buses |
| Dónde | 5 | Zona, jerarquía vial y corredores |
| Por qué | 6 | Causas y factores |
| Qué y a quién | 7 | Tipos de siniestro y vehículos más graves |
| Qué y a quién | 8 | Perfil de las víctimas y los más vulnerables |
| Qué y a quién | 9 | A quién afecta cada tipo de siniestro y vehículo |
| Consecuencias | 10 | Siniestros de alta consecuencia |

Todo está en este notebook: instalación de librerías, limpieza, funciones de apoyo y las diez preguntas. Cada pregunta tiene tres partes: el enunciado y la técnica, el código y la respuesta.

## 0. Instalación de las librerías

El notebook necesita **pandas, numpy, matplotlib, seaborn, openpyxl, pyarrow y folium**. La celda siguiente las instala en el entorno del notebook (si ya están, no hace nada). Si instala alguna, reinicia el kernel y vuelve a ejecutar desde el principio.

El archivo de datos `08. BDD_Septiembre_2026_SM.xlsx` debe estar en la misma carpeta que este notebook.

In [ ]:
# Instalación de las librerías (se ejecuta una sola vez)
%pip install -q "pandas>=2.2" "numpy>=1.26" "matplotlib>=3.8" "seaborn>=0.13" "openpyxl>=3.1" "pyarrow>=15" "folium>=0.17" ipython

## 1. Preparación

Este notebook es **autocontenido**: el código de limpieza y las funciones de apoyo están en las celdas siguientes, y las preguntas los usan igual que si fueran archivos aparte. Al ejecutarlo se crean las carpetas `datos/` (copia de los datos limpios) y `resultados/` (gráficos, tablas y mapas).

In [ ]:
import sys
import types
import warnings

warnings.filterwarnings("ignore")


def registrar_modulo(nombre, antes):
    """Convierte lo definido en una celda en un módulo importable (así las preguntas pueden hacer `from nombre import ...`)."""
    modulo = types.ModuleType(nombre)
    modulo.__dict__.update({k: v for k, v in globals().items() if k not in antes and not k.startswith("_")})
    sys.modules[nombre] = modulo

## 2. Limpieza y calidad de los datos

Antes de analizar, los datos se limpiaron en cuatro pasos (el código está en la celda siguiente). **No se eliminó ningún siniestro.**

1. **Estructura:** el Excel tenía una fila por siniestro, con la edad y el sexo de cada persona repartidos en muchas columnas. Se reorganizó con una fila por persona (37 096 filas).
2. **Fechas, horas y texto:** se corrigieron las horas escritas como texto y se unificó el formato.
3. **Validación:** no hay filas repetidas, coordenadas fuera de Ecuador ni víctimas que no coincidan con las personas registradas.
4. **Datos desconocidos:** donde el dataset decía «no identificado» o «ND», o ponía −1 en la edad, el dato se dejó vacío para que no cuente como un valor real.

**Ruido** son los datos que faltan y no se pueden usar. Como el análisis es retrospectivo, **no se rellenaron** esos vacíos: se usa solo lo que realmente se registró.

También se crearon métricas nuevas apoyadas en la literatura:

| Métrica | Referencia |
|---|---|
| `n_victimas, letalidad` | WHO (2023). Global status report on road safety 2023. Geneva: World Health Organization. ISBN 978-92-4-008651-7. | Elvik, R., Hoye, A., Vaa, T., Sorensen, M. (2009). The Handbook of Road Safety Measures (2a ed.). Emerald. |
| `severidad (FATAL / CON LESIONADOS / SIN VICTIMAS)` | AASHTO (2010). Highway Safety Manual (1a ed.). Escala de severidad tipo KABCO, simplificada a 3 niveles porque el dataset no distingue lesiones graves. |
| `severidad_ponderada` | AASHTO (2010). Highway Safety Manual: idea de 'accidentes equivalentes' (EPDO) que pondera mas a las victimas graves. Los pesos 10:1 usados aqui son un supuesto del proyecto. |
| `causa_grupo, factor_causal` | Haddon, W. (1968). The changing approach to the epidemiology, prevention, and amelioration of trauma. Am. J. Public Health, 58(8). | Sabey, B. E., Taylor, H. (1980). The known risks we run: the highway. TRRL Supplementary Report 567. |
| `usuario_vulnerable` | Peden, M. et al. (2004). World report on road traffic injury prevention. WHO. | WHO (2023). |
| `indice de estacionalidad` | Hyndman, R. J., Athanasopoulos, G. (2021). Forecasting: Principles and Practice (3a ed.). OTexts. |
| `uso de casco (contraste en P8)` | Liu, B. C. et al. (2008). Helmets for preventing injury in motorcycle riders. Cochrane Database of Systematic Reviews, 2008(1), CD004333. |

La **letalidad** de una persona o grupo se calcula como fallecidos / (fallecidos + lesionados). El **% de siniestros fatales** es el porcentaje de siniestros con al menos un fallecido.

In [ ]:
_antes = set(globals())      # (solo sirve para que las preguntas puedan escribir: from limpieza_datos import ...)

"""
limpieza_datos.py
=================
Limpieza de los datos, compartida por todas las preguntas del notebook.

Dataset: siniestros de transito en Ecuador, enero-septiembre 2026
(archivo "08. BDD_Septiembre_2026_SM.xlsx", hoja BDD_2026, 17.236 filas x 374 columnas).
Cada fila es un siniestro; las columnas *_1 ... *_43 describen hasta 43 participantes
(formato ancho) y TIPO_DE_VEHICULO_1 ... _13 hasta 13 vehiculos.

Que hace este modulo
--------------------
1. Lee el Excel y mide el RUIDO (datos que no sirven para el analisis).
2. Limpia y transforma: texto, fechas/horas, marcadores "ND"/"NO IDENTIFICADO"/-1, etc.
3. Pasa los participantes de formato ancho a formato largo (una fila por persona).
4. Crea metricas nuevas (features) basadas en literatura (ver REFERENCIAS).
5. Guarda el resultado en ./datos (cache parquet) para que cada pregunta cargue en segundos.
"""

import sys
from pathlib import Path

import numpy as np
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):          # evita errores de acentos en consolas de Windows
    sys.stdout.reconfigure(encoding="utf-8")

RAIZ = Path.cwd().resolve()          # carpeta del notebook: aquí debe estar el .xlsx y aquí se crean datos/ y resultados/
DIR_DATOS = RAIZ / "datos"
DIR_RES = RAIZ / "resultados"
N_PARTICIPANTES = 43
N_VEHICULOS = 13

# Pesos de la severidad ponderada (estilo EPDO). Son un SUPUESTO ajustable, no un estandar oficial.
PESO_FALLECIDO = 10
PESO_LESIONADO = 1

MESES = ["ENERO", "FEBRERO", "MARZO", "ABRIL", "MAYO", "JUNIO", "JULIO", "AGOSTO", "SEPTIEMBRE"]
DIAS = ["LUNES", "MARTES", "MIÉRCOLES", "JUEVES", "VIERNES", "SÁBADO", "DOMINGO"]
FRANJAS = ["Madrugada (00-05)", "Mañana (06-11)", "Tarde (12-17)", "Noche (18-23)"]
GRUPOS_EDAD = ["0-11", "12-17", "18-29", "30-64", "65+"]

# Agrupacion de las 26 causas probables (CODIGO_CAUSA) por factor contribuyente.
# Idea del modelo humano - vehiculo - via/entorno (Haddon 1968; Sabey y Taylor 1980).
CAUSA_GRUPO = {
    "C09": "Exceso de velocidad",
    "C14": "Distracción al conducir",
    "C06": "Alcohol / sustancias", "C07": "Alcohol / sustancias",
    "C03": "Fatiga / somnolencia",
    "C23": "Irrespeto de señales / prioridad", "C24": "Irrespeto de señales / prioridad",
    "C25": "Irrespeto de señales / prioridad", "C26": "Irrespeto de señales / prioridad",
    "C11": "Maniobras y distancia", "C12": "Maniobras y distancia", "C18": "Maniobras y distancia",
    "C19": "Maniobras y distancia", "C22": "Maniobras y distancia",
    "C16": "Conducta del peatón / pasajero", "C17": "Conducta del peatón / pasajero",
    "C27": "Conducta del peatón / pasajero",
    "C04": "Vehículo / carga", "C05": "Vehículo / carga", "C08": "Vehículo / carga",
    "C01": "Vía y entorno", "C02": "Vía y entorno", "C10": "Vía y entorno", "C21": "Vía y entorno",
    "C15": "Otras infracciones", "C20": "Otras infracciones",
}
CAUSA_FACTOR = {c: ("Vehículo" if g == "Vehículo / carga" else "Vía y entorno" if g == "Vía y entorno" else "Humano")
                for c, g in CAUSA_GRUPO.items()}

REFERENCIAS = {
    "n_victimas, letalidad": "WHO (2023). Global status report on road safety 2023. Geneva: World Health Organization. "
                             "ISBN 978-92-4-008651-7. | Elvik, R., Hoye, A., Vaa, T., Sorensen, M. (2009). "
                             "The Handbook of Road Safety Measures (2a ed.). Emerald.",
    "severidad (FATAL / CON LESIONADOS / SIN VICTIMAS)": "AASHTO (2010). Highway Safety Manual (1a ed.). Escala de severidad "
                             "tipo KABCO, simplificada a 3 niveles porque el dataset no distingue lesiones graves.",
    "severidad_ponderada": "AASHTO (2010). Highway Safety Manual: idea de 'accidentes equivalentes' (EPDO) que pondera "
                           "mas a las victimas graves. Los pesos 10:1 usados aqui son un supuesto del proyecto.",
    "causa_grupo, factor_causal": "Haddon, W. (1968). The changing approach to the epidemiology, prevention, and amelioration "
                                  "of trauma. Am. J. Public Health, 58(8). | Sabey, B. E., Taylor, H. (1980). The known risks "
                                  "we run: the highway. TRRL Supplementary Report 567.",
    "usuario_vulnerable": "Peden, M. et al. (2004). World report on road traffic injury prevention. WHO. | WHO (2023).",
    "indice de estacionalidad": "Hyndman, R. J., Athanasopoulos, G. (2021). Forecasting: Principles and Practice (3a ed.). OTexts.",
    "uso de casco (contraste en P8)": "Liu, B. C. et al. (2008). Helmets for preventing injury in motorcycle riders. Cochrane "
                                      "Database of Systematic Reviews, 2008(1), CD004333.",
}

# Columnas de analisis (para medir ruido). 'ND' en campos viales = no aplica (el dataset solo tiene
# datos de vias de la red estatal); se reporta aparte y no cuenta como ruido.
CAMPOS_SIN_RUIDO = ["fecha", "hora_num", "provincia", "canton", "parroquia", "zona", "lat", "lon", "causa_probable",
                    "tipo_siniestro", "tipo_veh_1", "servicio_1", "lesionados", "fallecidos", "ente_control"]
CAMPOS_VIALES = ["id_via", "nombre_via", "ubicacion_via", "jerarquia_via"]
CAMPOS_PART = ["edad", "sexo", "condicion", "rol", "tipo_id"]


# --------------------------------------------------------------------------------------
# Lectura
# --------------------------------------------------------------------------------------
def _ruta_excel():
    archivos = [p for p in RAIZ.glob("*.xlsx") if not p.name.startswith("~$")]
    if not archivos:
        raise FileNotFoundError("No se encontro el .xlsx en la carpeta del proyecto: " + str(RAIZ))
    return archivos[0]


def _leer_crudo():
    return pd.read_excel(_ruta_excel(), engine="openpyxl")


# --------------------------------------------------------------------------------------
# Limpieza + transformacion
# --------------------------------------------------------------------------------------
def _texto(s):
    """Quita espacios sobrantes y pasa a mayusculas (corrige 'colisión' vs 'COLISIÓN')."""
    return s.astype("string").str.strip().str.upper()


def _construir(raw):
    pasos = []   # (paso, descripcion, registros_o_celdas_afectados, porcentaje)

    def anotar(paso, descr, n, total):
        pasos.append({"paso": paso, "descripcion": descr, "afectados": int(n),
                      "base": int(total), "porcentaje": round(100 * n / total, 2) if total else 0.0})

    n = len(raw)
    anotar("0. Lectura", f"Filas x columnas leídas: {n} x {raw.shape[1]}", 0, n)

    # Paso 1 - estructura: columnas casi vacias (slots de participantes/vehiculos poco usados)
    vacias = raw.isna().mean()
    anotar("1. Estructura", "Columnas con >95% de celdas vacías (slots de participantes/vehículos poco usados); "
                            "no son ruido: se reorganizan en formato largo", (vacias > 0.95).sum(), raw.shape[1])
    anotar("1. Estructura", "Celdas vacías sobre el total de la tabla ancha (esparsidad estructural)",
           raw.isna().sum().sum(), raw.size)

    # Paso 2 - duplicados
    anotar("2. Duplicados", "Filas completamente duplicadas", raw.duplicated().sum(), n)
    ids_rep = raw["SINIESTROS"].duplicated(keep=False)
    anotar("2. Duplicados", "Registros con código SINIESTROS repetido (son hechos distintos: otra provincia/ente/"
                            "coordenadas; se conservan y se usa un identificador interno 'sid')", ids_rep.sum(), n)

    # Paso 3 - renombrar y seleccionar columnas
    s = pd.DataFrame({"sid": np.arange(n)})
    s["id_siniestro"] = raw["SINIESTROS"].astype("string")
    s["fecha"] = pd.to_datetime(raw["FECHA"])

    # Paso 4 - hora: objetos time y 5 textos 'HH:MM'
    hs = raw["HORA"].map(lambda x: x.strftime("%H:%M") if hasattr(x, "strftime") else str(x)[:5])
    n_txt = raw["HORA"].map(lambda x: isinstance(x, str)).sum()
    anotar("4. Fecha/hora", "Horas guardadas como texto 'HH:MM' en lugar de hora; se convierten", n_txt, n)
    s["hora_num"] = hs.str[:2].astype(int)
    s["minuto"] = hs.str[3:5].astype(int)
    s["hora_decimal"] = s["hora_num"] + s["minuto"] / 60
    incons = (s["hora_num"] != raw["PERIODO_2"]).sum()
    anotar("4. Fecha/hora", "Hora inconsistente con la franja PERIODO_2 tras la conversión", incons, n)
    anotar("4. Fecha/hora", "Día de la semana (DIA_2) inconsistente con FECHA",
           ((s["fecha"].dt.day_of_week + 1) != raw["DIA_2"]).sum(), n)

    s["mes"] = s["fecha"].dt.month
    s["dia_semana"] = s["fecha"].dt.day_of_week + 1        # 1 = lunes ... 7 = domingo
    s["feriado"] = _texto(raw["FERIADO"]).eq("SI")

    # Paso 5 - texto
    for nuevo, viejo in [("ente_control", "ENTE_DE_CONTROL"), ("provincia", "PROVINCIA"), ("canton", "CANTON"),
                         ("parroquia", "PARROQUIA"), ("zona_planificacion", "ZONA_PLANIFICACION"), ("zona", "ZONA"),
                         ("tipo_siniestro", "TIPO_DE_SINIESTRO"), ("causa_probable", "CAUSA_PROBABLE"),
                         ("tipo_veh_1", "TIPO_DE_VEHICULO_1"), ("servicio_1", "SERVICIO_1"),
                         ("tipo_veh_2", "TIPO_DE_VEHICULO_2")]:
        s[nuevo] = _texto(raw[viejo])
    anotar("5. Texto", "Categorías de tipo de siniestro con variantes de mayúsculas/espacios unificadas",
           (raw["TIPO_DE_SINIESTRO"].astype("string") != s["tipo_siniestro"]).sum(), n)
    s["cod_causa"] = _texto(raw["CODIGO_CAUSA"])
    s["direccion"] = _texto(raw["DIRECCION"])
    s["lat"] = raw["LATITUD_Y"].astype(float)
    s["lon"] = raw["LONGITUD_X"].astype(float)

    # Paso 6 - coordenadas: dentro de Ecuador (continente + Galapagos)
    fuera = ~(s["lat"].between(-5.1, 1.6) & s["lon"].between(-92.2, -75.0)) | s["lat"].isna() | s["lon"].isna()
    anotar("6. Coordenadas", "Coordenadas fuera de Ecuador (continental + Galápagos) o nulas", fuera.sum(), n)
    s.loc[fuera, ["lat", "lon"]] = np.nan

    # Paso 7 - campos viales: ND = no aplica
    for nuevo, viejo in [("id_via", "ID_DE_LA_VIA"), ("nombre_via", "NOMBRE_DE_LA_VIA"),
                         ("ubicacion_via", "UBICACION_DE_LA_VIA"), ("jerarquia_via", "JERARQUIA_DE_LA_VIA")]:
        s[nuevo] = _texto(raw[viejo].astype("string")).replace({"ND": pd.NA, "0": pd.NA})
    s["via_estatal"] = s["ubicacion_via"].notna()
    anotar("7. Vías", "Registros sin datos de vía ('ND' o 0): ocurren fuera de la red vial estatal (no aplica)",
           (~s["via_estatal"]).sum(), n)

    # Paso 8 - vehiculos
    mapa_veh = {"AUTOMOVIL": "automovil", "BICICLETA": "bicicleta", "BUS": "bus", "CAMION": "camion",
                "CAMIONETA": "camioneta", "EMERGENCIAS": "emergencias", "ESPECIAL": "especial",
                "FURGONETA": "furgoneta", "MOTOCICLETA": "motocicleta", "NO_IDENTIFICADO": "no_identificado",
                "SCOOTER_ELECTRICO": "scooter", "TRICIMOTO": "tricimoto", "VEHICULO_DEPORTIVO_UTILITARIO": "suv"}
    for viejo, nuevo in mapa_veh.items():
        s["n_" + nuevo] = raw[viejo].astype(int)
    s["n_vehiculos"] = raw["SUMA_DE_VEHICULOS"].astype(int)
    anotar("8. Vehículos", "Registros donde SUMA_DE_VEHICULOS != suma de columnas por tipo",
           (s[["n_" + v for v in mapa_veh.values()]].sum(axis=1) != s["n_vehiculos"]).sum(), n)
    serv = pd.concat([_texto(raw[f"SERVICIO_{i}"]) for i in range(1, N_VEHICULOS + 1)], axis=1)
    s["transporte_publico"] = serv.eq("PÚBLICO").any(axis=1)

    # Paso 9 - victimas y validacion cruzada con participantes
    s["lesionados"] = raw["LESIONADOS"].astype(int)
    s["fallecidos"] = raw["FALLECIDOS"].astype(int)
    cond = pd.concat([_texto(raw[f"CONDICION_{i}"]) for i in range(1, N_PARTICIPANTES + 1)], axis=1)
    ok = ((cond.eq("FALLECIDO").sum(axis=1) == s["fallecidos"]) &
          (cond.eq("LESIONADO").sum(axis=1) == s["lesionados"]))
    anotar("9. Validación", "Siniestros cuyas víctimas (FALLECIDOS/LESIONADOS) NO coinciden con las condiciones "
                            "de los participantes", (~ok).sum(), n)
    anotar("9. Validación", "Siniestros con >10 lesionados (atípicos reales; se conservan y se marcan)",
           (s["lesionados"] > 10).sum(), n)

    # ---------------- Features (metricas nuevas) ----------------
    s["n_victimas"] = s["fallecidos"] + s["lesionados"]
    s["severidad"] = np.select([s["fallecidos"] > 0, s["lesionados"] > 0], ["FATAL", "CON LESIONADOS"], "SIN VÍCTIMAS")
    s["severidad"] = pd.Categorical(s["severidad"], ["SIN VÍCTIMAS", "CON LESIONADOS", "FATAL"], ordered=True)
    s["fatal"] = s["fallecidos"] > 0
    s["letalidad"] = np.where(s["n_victimas"] > 0, s["fallecidos"] / s["n_victimas"].replace(0, np.nan), np.nan)
    s["severidad_ponderada"] = PESO_FALLECIDO * s["fallecidos"] + PESO_LESIONADO * s["lesionados"]
    s["franja"] = pd.cut(s["hora_num"], [-1, 5, 11, 17, 23], labels=FRANJAS).astype("object")
    s["franja"] = pd.Categorical(s["franja"], FRANJAS, ordered=True)
    s["nocturno"] = (s["hora_num"] < 6) | (s["hora_num"] >= 18)
    s["fin_semana"] = s["dia_semana"] >= 6
    s["tipo_dia"] = np.select([s["feriado"], s["fin_semana"]], ["FERIADO", "FIN DE SEMANA"], "LABORABLE")
    s["mes_nombre"] = pd.Categorical(s["mes"].map(lambda m: MESES[m - 1]), MESES, ordered=True)
    s["dia_nombre"] = pd.Categorical(s["dia_semana"].map(lambda d: DIAS[d - 1]), DIAS, ordered=True)
    s["causa_grupo"] = s["cod_causa"].map(CAUSA_GRUPO).fillna("Otras infracciones")
    s["factor_causal"] = s["cod_causa"].map(CAUSA_FACTOR).fillna("Humano")
    s["involucra_moto"] = s["n_motocicleta"] > 0
    s["solo_moto"] = s["n_motocicleta"] == s["n_vehiculos"]
    vru = s[["n_motocicleta", "n_bicicleta", "n_scooter", "n_tricimoto"]].sum(axis=1) > 0
    s["usuario_vulnerable"] = vru | s["tipo_siniestro"].eq("ATROPELLOS")
    s["solo_vehiculos_cerrados"] = s[["n_motocicleta", "n_bicicleta", "n_scooter", "n_tricimoto",
                                      "n_no_identificado"]].sum(axis=1) == 0

    # ---------------- Participantes: formato ancho -> largo ----------------
    partes = []
    for i in range(1, N_PARTICIPANTES + 1):
        t = pd.DataFrame({"sid": s["sid"], "orden": i,
                          "tipo_id": _texto(raw[f"TIPO_ID_{i}"]), "edad": raw[f"EDAD_{i}"].astype(float),
                          "sexo": _texto(raw[f"SEXO_{i}"]), "condicion": _texto(raw[f"CONDICION_{i}"]),
                          "rol": _texto(raw[f"PARTICIPANTE_{i}"]), "casco": _texto(raw[f"CASCO_{i}"]),
                          "cinturon": _texto(raw[f"CINTURON_{i}"])})
        partes.append(t[t["rol"].notna()])
    p = pd.concat(partes, ignore_index=True)
    anotar("10. Participantes", "Participantes (filas) tras pasar a formato largo", len(p), len(p))

    p["edad"] = p["edad"].where(p["edad"] >= 0)                 # -1 = sin dato
    for c in ["sexo", "condicion", "tipo_id"]:
        p[c] = p[c].replace({"NO IDENTIFICADO": pd.NA})
    p["grupo_edad"] = pd.cut(p["edad"], [-1, 11, 17, 29, 64, 200], labels=GRUPOS_EDAD).astype("object")
    p["grupo_edad"] = pd.Categorical(p["grupo_edad"], GRUPOS_EDAD, ordered=True)
    p["victima"] = p["condicion"].isin(["FALLECIDO", "LESIONADO"])
    cols = ["sid", "fecha", "hora_num", "franja", "nocturno", "mes", "dia_semana", "tipo_dia", "provincia", "canton",
            "zona", "ente_control", "tipo_siniestro", "causa_grupo", "n_vehiculos", "solo_moto",
            "solo_vehiculos_cerrados", "usuario_vulnerable"]
    p = p.merge(s[cols], on="sid", how="left")

    # ---------------- Ruido ----------------
    filas = []
    for c in CAMPOS_SIN_RUIDO:
        base = s[c]
        malos = base.isna()
        if not pd.api.types.is_numeric_dtype(base) and not pd.api.types.is_datetime64_any_dtype(base):
            malos = malos | base.astype("string").isin(["ND", "NO IDENTIFICADO"])
        filas.append({"tabla": "siniestros", "campo": c, "inutilizables": int(malos.sum()), "base": n,
                      "tipo": "ruido"})
    for c in CAMPOS_VIALES:
        filas.append({"tabla": "siniestros", "campo": c, "inutilizables": int(s[c].isna().sum()), "base": n,
                      "tipo": "no aplica"})
    for c in CAMPOS_PART:
        malos = p[c].isna()
        filas.append({"tabla": "participantes", "campo": c, "inutilizables": int(malos.sum()), "base": len(p),
                      "tipo": "ruido"})
    ruido = pd.DataFrame(filas)
    ruido["porcentaje"] = (100 * ruido["inutilizables"] / ruido["base"]).round(2)
    return s, p, pd.DataFrame(pasos), ruido


def _resumen_ruido(ruido):
    r = ruido[ruido["tipo"] == "ruido"]
    total = {}
    for tabla, g in r.groupby("tabla"):
        total[tabla] = float(round(100 * g["inutilizables"].sum() / g["base"].sum(), 2))
    tot = float(round(100 * r["inutilizables"].sum() / r["base"].sum(), 2))
    return total, tot


def reconstruir():
    raw = _leer_crudo()
    s, p, pasos, ruido = _construir(raw)
    DIR_DATOS.mkdir(exist_ok=True)
    DIR_RES.mkdir(exist_ok=True)
    s.to_parquet(DIR_DATOS / "siniestros_limpio.parquet", index=False)
    p.to_parquet(DIR_DATOS / "participantes_limpio.parquet", index=False)
    pasos.to_csv(DIR_RES / "limpieza_pasos.csv", index=False, encoding="utf-8-sig")
    ruido.to_csv(DIR_RES / "limpieza_ruido_por_campo.csv", index=False, encoding="utf-8-sig")
    return s, p, pasos, ruido


def cargar_datos():
    """Devuelve (siniestros, participantes) limpios. Usa la cache si esta al dia."""
    fs, fp = DIR_DATOS / "siniestros_limpio.parquet", DIR_DATOS / "participantes_limpio.parquet"
    vigente = fs.exists() and fp.exists() and min(fs.stat().st_mtime, fp.stat().st_mtime) > _ruta_excel().stat().st_mtime
    if not vigente:
        print("Construyendo datos limpios (solo la primera vez, ~1 min)...")
        s, p, _, _ = reconstruir()
    else:
        s, p = pd.read_parquet(fs), pd.read_parquet(fp)
    DIR_RES.mkdir(exist_ok=True)
    return s, p


registrar_modulo("limpieza_datos", _antes)

Con las funciones anteriores se cargan los datos limpios y se muestran los pasos de limpieza y el ruido por campo (la primera vez, construir los datos tarda alrededor de 1 minuto).

In [ ]:
import pandas as pd
from IPython.display import display

sin, part = cargar_datos()          # siniestros (1 fila = 1 siniestro) y participantes (1 fila = 1 persona)
if not (DIR_RES / "limpieza_pasos.csv").exists():
    reconstruir()                   # genera los reportes de pasos y ruido si aún no existen

print(f"Siniestros: {sin.shape[0]:,} filas x {sin.shape[1]} columnas | Participantes: {part.shape[0]:,} filas x {part.shape[1]} columnas")
print("\nPasos de limpieza y transformación:")
display(pd.read_csv(DIR_RES / "limpieza_pasos.csv"))
print("Ruido por campo (celdas inutilizables):")
display(pd.read_csv(DIR_RES / "limpieza_ruido_por_campo.csv"))

## 3. Funciones de apoyo

Estilo de los gráficos (con la paleta del proyecto), guardado de figuras, tablas y mapas de Folium, tabla de gravedad por grupo y clasificación Pareto ABC. Cada figura se guarda en `resultados/` y también se muestra en el notebook.

Los mapas (preguntas 3 y 4) se guardan como HTML y, si hay Edge o Chrome instalado, también como imagen; necesitan conexión a internet para el mapa base.

In [ ]:
_antes = set(globals())      # (solo sirve para que las preguntas puedan escribir: from utilidades import ...)

"""
utilidades.py
=============
Funciones auxiliares compartidas por las preguntas: estilo de gráficos, guardado de
resultados (figuras, tablas y mapas de Folium), tabla de gravedad por grupo y clasificación Pareto ABC.
"""

import html
import shutil
import subprocess
import tempfile
from pathlib import Path

from IPython.display import Image, display

import folium
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


# Paleta categórica (orden fijo) y rampas: azul secuencial; divergente azul-gris-naranja
PALETA = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRIS = "#8a8a85"
CMAP_SEC = LinearSegmentedColormap.from_list("sec", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"])
CMAP_DIV = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f1f0ec", "#eb6834"])


def estilo():
    sns.set_theme(style="whitegrid", context="notebook", palette=PALETA)
    plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.titleweight": "bold",
                         "axes.labelsize": 10, "grid.color": "#e6e5e0", "grid.linewidth": 0.7,
                         "axes.spines.top": False, "axes.spines.right": False})


estilo()


def guardar_fig(fig, nombre):
    DIR_RES.mkdir(exist_ok=True)
    fig.tight_layout()
    ruta = DIR_RES / f"{nombre}.png"
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[figura] {ruta.name}")
    display(Image(filename=str(ruta)))          # en el notebook, además de guardarla, se muestra


# Mapa base gris claro de Esri (no requiere clave de API); las capas de datos se ven mejor sobre él
TILES_GRIS = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ATRIB_GRIS = "Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ"


def mapa_base(**kw):
    """Mapa de Folium con el fondo gris de Esri (sin entrada en el control de capas)."""
    kw.setdefault("zoom_snap", 0.25)
    m = folium.Map(tiles=None, **kw)
    folium.TileLayer(TILES_GRIS, attr=ATRIB_GRIS, name="Mapa base", control=False).add_to(m)
    return m


def etiqueta_mapa(m, html, abajo=False):
    """Cuadro de título del mapa, arriba a la izquierda (o abajo si la parte de arriba lleva una escala de color)."""
    pos = "bottom:45px;left:10px" if abajo else "top:10px;left:55px"
    m.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;{pos};z-index:9999;background:white;padding:5px 9px;'
        f'border-radius:4px;font:13px Arial;box-shadow:0 0 4px #888">{html}</div>'))


def _navegador():
    """Ruta de Edge o Chrome (para capturar el mapa en PNG), o None si no hay ninguno."""
    candidatos = [r"C:\Program Files (x86)\Microsoft\Edge\Application\msedge.exe",
                  r"C:\Program Files\Microsoft\Edge\Application\msedge.exe",
                  r"C:\Program Files\Google\Chrome\Application\chrome.exe",
                  r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe"]
    for c in candidatos + [shutil.which(x) for x in ("msedge", "chrome", "google-chrome", "chromium")]:
        if c and Path(c).exists():
            return c
    return None


def guardar_mapa(mapa, nombre, ancho=900, alto=900):
    """Guarda un mapa de Folium como HTML interactivo y, si hay Edge o Chrome, también como PNG.
    Necesita internet: el HTML carga Leaflet y las teselas del mapa base desde la red."""
    DIR_RES.mkdir(exist_ok=True)
    ruta_html = DIR_RES / f"{nombre}.html"
    if isinstance(mapa, str):                 # página HTML ya armada (facetas)
        ruta_html.write_text(mapa, encoding="utf-8")
    else:
        mapa.save(str(ruta_html))
    print(f"[mapa]   {ruta_html.name}")
    nav = _navegador()
    if nav is None:
        print(f"[aviso]  No se encontró Edge ni Chrome: no se creó {nombre}.png (abre el HTML y toma una captura).")
        return
    ruta_png = DIR_RES / f"{nombre}.png"
    ruta_png.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as perfil:
        try:
            subprocess.run([nav, "--headless=new", "--disable-gpu", "--hide-scrollbars", f"--user-data-dir={perfil}",
                            f"--window-size={ancho},{alto}", "--force-device-scale-factor=2", "--virtual-time-budget=25000",
                            f"--screenshot={ruta_png}", ruta_html.as_uri()],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=180)
        except subprocess.TimeoutExpired:
            pass
    print(f"[figura] {ruta_png.name}" if ruta_png.exists() else f"[aviso]  No se pudo crear {ruta_png.name}")
    if ruta_png.exists():
        display(Image(filename=str(ruta_png)))


def guardar_facetas(mapas, nombre, ancho, alto):
    """Une varios mapas de Folium en una sola página (una faceta por mapa, lado a lado) y la guarda como HTML y PNG."""
    celdas = "".join(f'<iframe srcdoc="{html.escape(m.get_root().render(), quote=True)}"></iframe>' for m in mapas)
    pagina = ("<!DOCTYPE html><html><head><meta charset='utf-8'><style>html,body{margin:0;height:100%}"
              "body{display:flex;gap:6px;background:#fff}iframe{flex:1;border:0;height:100%}</style></head>"
              f"<body>{celdas}</body></html>")
    guardar_mapa(pagina, nombre, ancho, alto)


def guardar_tabla(df, nombre, indice=True):
    DIR_RES.mkdir(exist_ok=True)
    ruta = DIR_RES / f"{nombre}.csv"
    df.to_csv(ruta, index=indice, encoding="utf-8-sig")
    print(f"[tabla]  {ruta.name}")


def seccion(texto):
    print("\n" + "=" * 90 + f"\n{texto}\n" + "=" * 90)


def mostrar(df, n=15, **kw):
    with pd.option_context("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 60,
                           "display.float_format", lambda x: f"{x:,.2f}"):
        print(df.head(n).to_string(**kw))


def gravedad(df, por, minimo=0):
    """Tabla de gravedad por grupo: siniestros, con fallecidos (% fatales), fallecidos, lesionados y víctimas por siniestro."""
    g = df.groupby(por, observed=True).agg(siniestros=("sid", "size"), fatales=("fatal", "sum"),
                                          fallecidos=("fallecidos", "sum"), lesionados=("lesionados", "sum"))
    g["% fatales"] = 100 * g["fatales"] / g["siniestros"]
    g["víctimas por siniestro"] = (g["fallecidos"] + g["lesionados"]) / g["siniestros"]
    return g[g["siniestros"] >= minimo]


def pareto_abc(serie, cortes=(0.80, 0.95)):
    """Clasificación ABC (80-15-5) de una serie de valores por categoría."""
    d = serie.sort_values(ascending=False).to_frame("valor")
    d["%"] = 100 * d["valor"] / d["valor"].sum()
    d["% acumulado"] = d["%"].cumsum()
    previo = d["% acumulado"] - d["%"]          # acumulado antes de sumar la categoría
    d["clase"] = np.where(previo < 100 * cortes[0], "A", np.where(previo < 100 * cortes[1], "B", "C"))
    return d


registrar_modulo("utilidades", _antes)

---
## Pregunta 1 · Cuándo: ¿Cuándo ocurren los siniestros?

> **¿En qué meses, días de la semana y horas ocurren más siniestros, y ese patrón cambia entre zona urbana y rural?**

**Técnicas:** índices de estacionalidad (valor / promedio; mayor a 1 = temporada alta) y gráficos de barras y líneas: uno general y uno por categoría (mes, día, hora) urbana vs rural.

**Qué hace el código**

- Cuenta los días del periodo (273) para que un mes largo no gane solo por tener más días.
- `indices()` calcula los siniestros por día calendario en cada mes y día de la semana, y por hora, y los divide entre su promedio: **1 = promedio, mayor a 1 = temporada alta**. Se aplica a todos los siniestros, a los urbanos y a los rurales (cada zona contra su propio promedio) y guarda `p01_indices.csv`.
- Dibuja una imagen general (mes, día, hora) y una por categoría con urbana (azul) vs rural (naranja).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from limpieza_datos import cargar_datos, MESES, DIAS
from utilidades import PALETA, guardar_fig, guardar_tabla, mostrar, seccion

sin, _ = cargar_datos()
dias_cal = pd.Series(pd.date_range(sin["fecha"].min(), sin["fecha"].max()))   # días del periodo observado
dias_por_mes = dias_cal.dt.month.value_counts().sort_index()
dias_por_dow = dias_cal.dt.day_of_week.add(1).value_counts().sort_index()


def indices(df):
    """Índice = siniestros por día calendario / promedio de la categoría (1 = promedio)."""
    por_mes = df.groupby("mes").size() / dias_por_mes
    por_dia = df.groupby("dia_semana").size() / dias_por_dow
    por_hora = df.groupby("hora_num").size().reindex(range(24), fill_value=0)
    return {"mes": (por_mes / por_mes.mean()).rename(index=lambda m: MESES[m - 1]),
            "día": (por_dia / por_dia.mean()).rename(index=lambda d: DIAS[d - 1]),
            "hora": por_hora / por_hora.mean()}


# 1) Índices generales y por zona
seccion("1) Índices de estacionalidad")
gen = indices(sin)
zonas = {"Urbana": indices(sin[sin["zona"] == "URBANA"]), "Rural": indices(sin[sin["zona"] == "RURAL"])}
for cat in ["mes", "día", "hora"]:
    tabla = pd.DataFrame({"General": gen[cat], **{z: zonas[z][cat] for z in zonas}})
    print(f"\nÍndice por {cat}:")
    mostrar(tabla.round(2), 24)
tablas = {cat: pd.DataFrame({"General": gen[cat], **{z: zonas[z][cat] for z in zonas}}) for cat in gen}
guardar_tabla(pd.concat(tablas, names=["categoria", "valor"]), "p01_indices")
print("\nHoras pico generales (índice > 1.3):", [int(h) for h in gen["hora"][gen["hora"] > 1.3].index])
for cat in ["mes", "día", "hora"]:
    for z, ind in zonas.items():
        print(f"{cat} | {z}: máximo {ind[cat].idxmax()} ({ind[cat].max():.2f}), mínimo {ind[cat].idxmin()} ({ind[cat].min():.2f})")

# 2) Imagen general: las tres categorías (naranja = temporada alta)
fig, ax = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [9, 7, 24]})
for a, cat, xl in [(ax[0], "mes", "Mes (2026)"), (ax[1], "día", "Día de la semana"), (ax[2], "hora", "Hora de ocurrencia (0-23 h)")]:
    idx = gen[cat]
    a.bar(range(len(idx)), idx.values, color=[PALETA[1] if v > 1 else PALETA[0] for v in idx.values], width=0.75)
    a.axhline(1, color="#52514e", lw=1)
    a.set_xticks(range(len(idx)))
    a.set_xticklabels([str(i)[:3].title() if cat != "hora" else str(i) for i in idx.index], rotation=0 if cat == "hora" else 45)
    a.set_title(f"Índice por {cat if cat != 'hora' else 'hora del día'}")
    a.set_xlabel(xl)
    a.set_ylabel("Índice (1 = promedio)")
guardar_fig(fig, "p01_indices_general")

# 3) Una imagen por categoría: urbana vs rural (cada zona contra su propio promedio)
seccion("3) Urbana vs rural")
COL = {"Urbana": PALETA[0], "Rural": PALETA[1]}
for cat, xl, nombre in [("mes", "Mes (2026)", "p01_indices_mes_zona"), ("día", "Día de la semana", "p01_indices_dia_zona"),
                        ("hora", "Hora de ocurrencia (0-23 h)", "p01_indices_hora_zona")]:
    fig, a = plt.subplots(figsize=(11 if cat == "hora" else 8, 4.2))
    n = len(gen[cat])
    x = np.arange(n)
    for k, (z, ind) in enumerate(zonas.items()):
        if cat == "hora":
            a.plot(x, ind[cat].values, color=COL[z], lw=2, marker="o", ms=4, label=z)
        else:
            a.bar(x + (k - 0.5) * 0.38, ind[cat].values, width=0.36, color=COL[z], label=z)
    a.axhline(1, color="#52514e", lw=1, ls="--")
    a.set_xticks(x)
    a.set_xticklabels([str(i)[:3].title() if cat != "hora" else str(i) for i in gen[cat].index])
    a.set_title(f"Índice por {cat if cat != 'hora' else 'hora del día'}: zona urbana vs rural")
    a.set_xlabel(xl)
    a.set_ylabel("Índice (1 = promedio de su zona)")
    a.legend(frameon=False, ncol=2, loc="upper left")
    guardar_fig(fig, nombre)

### Respuesta a la pregunta 1

**Resultados**

| Índice | General | Urbana | Rural |
|---|---|---|---|
| Domingo | **1.38** | 1.31 | **1.50** |
| 19 h | **1.59** | 1.43 | **1.86** |
| 7 h | 1.20 | 1.15 | 1.30 |
| 0 a 2 h | 0.68 a 0.79 | 0.72 a 0.87 | 0.54 a 0.65 |
| Enero → agosto | 0.82 → 1.15 | 0.83 → 1.13 | 0.78 → 1.17 |

Horas con índice mayor a 1.3 (general): 17, 18, 19 y 20 h. El día más bajo es martes (urbana 0.81) o miércoles (rural 0.75).

**Análisis**

1. **El patrón es el mismo en ambas zonas:** fin de semana, tarde-noche (17 a 20 h, máximo a las 19 h) y alza de enero a agosto.
2. **La zona rural es más extrema** (domingo, 19 h y pico matutino de 6 y 7 h); **la urbana es más pareja** y tiene más actividad de madrugada.

**Limitaciones**

- Cada zona se compara con su propio promedio: el índice dice *cuándo*, no *cuánto*. La urbana tiene más siniestros (10 939 frente a 6 297).
- Nueve meses no permiten hablar de estacionalidad anual; el alza mensual mezcla estacionalidad y tendencia (posible mayor cobertura del registro, no comprobada).

---
## Pregunta 2 · Cuándo: Días críticos

> **¿Qué días tuvieron muchos más siniestros o fallecidos de lo habitual y qué tienen en común (día de la semana, feriado, mes, causa dominante)?**

**Técnicas:** valores estandarizados (z-score) de la serie diaria, como en el Ejemplo 1; correlación entre siniestros y fallecidos; comparación de los días críticos con el total de días (día de la semana, mes, feriado y causa dominante) en gráficos facetados.

**Qué hace el código**

- Arma la serie diaria de siniestros y fallecidos (con los días en 0) y marca el día de la semana, el fin de semana y el feriado.
- Mide **cuántas desviaciones estándar se aparta cada día de lo habitual**, donde «habitual» es el promedio de su mismo día de la semana (así no se marca como crítico cada domingo solo por ser domingo). Cero es un día normal; un día es **crítico** si pasa de 2 en siniestros o en fallecidos. Guarda `p02_dias_criticos.csv`.
- Grafica cada día como un punto (rojo si es crítico, con su fecha) y luego compara, en cuatro paneles, cómo se reparten los días críticos y **todos** los días según día de la semana, mes, feriado y causa dominante.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from limpieza_datos import DIAS, cargar_datos
from utilidades import GRIS, PALETA, guardar_fig, guardar_tabla, mostrar, seccion

sin, _ = cargar_datos()
dias = pd.date_range(sin["fecha"].min(), sin["fecha"].max())
d = pd.DataFrame({"siniestros": sin.groupby("fecha").size(), "fallecidos": sin.groupby("fecha")["fallecidos"].sum()}
                 ).reindex(dias, fill_value=0)
d["dow"] = d.index.day_of_week
d["fin_semana"] = d["dow"] >= 5
d["feriado"] = d.index.isin(sin.loc[sin["feriado"], "fecha"].unique())

# 1) Estandarización: cada día se compara contra el promedio de su mismo día de la semana,
#    para no marcar como "crítico" cada domingo solo por ser domingo.
#    z = cuántas desviaciones estándar está un día por encima (o por debajo) de lo habitual; 0 = día normal.
seccion("1) Serie diaria estandarizada (z-score)")
for c in ["siniestros", "fallecidos"]:
    d[f"z_{c}"] = (d[c] - d.groupby("dow")[c].transform("mean")) / d.groupby("dow")[c].transform("std")
print(f"Correlación entre siniestros y fallecidos por día: {d['siniestros'].corr(d['fallecidos']):.2f}")
print(f"Días con z > 2: siniestros {int((d['z_siniestros'] > 2).sum())} | fallecidos {int((d['z_fallecidos'] > 2).sum())}")

# 2) Días críticos y sus características
seccion("2) Días críticos (z > 2 en siniestros o fallecidos)")
d["critico"] = (d["z_siniestros"] > 2) | (d["z_fallecidos"] > 2)
cr = d[d["critico"]].copy()
cr["causa_dominante"] = sin.groupby("fecha")["causa_grupo"].agg(lambda s: s.value_counts().index[0]).reindex(cr.index)
cr["dia"] = [DIAS[i].title() for i in cr["dow"]]
cols = ["dia", "siniestros", "fallecidos", "z_siniestros", "z_fallecidos", "feriado", "causa_dominante"]
mostrar(cr[cols].round(2), 40)
guardar_tabla(cr[cols], "p02_dias_criticos")
print(f"\n{len(cr)} días críticos de {len(d)} ({100 * len(cr) / len(d):.1f}%)")
for var in ["fin_semana", "feriado"]:
    print(f"% de días críticos entre los días '{var}': {100 * d.loc[d[var], 'critico'].mean():.1f}% | resto: {100 * d.loc[~d[var], 'critico'].mean():.1f}%")
print("Críticos por mes:", cr.groupby(cr.index.month).size().to_dict())

# Composición de causas en días críticos frente al resto
comp = pd.concat([sin[sin["fecha"].isin(cr.index)]["causa_grupo"].value_counts(normalize=True).rename("críticos"),
                  sin[~sin["fecha"].isin(cr.index)]["causa_grupo"].value_counts(normalize=True).rename("resto")], axis=1) * 100
comp["diferencia (pp)"] = comp["críticos"] - comp["resto"]
mostrar(comp.sort_values("diferencia (pp)", ascending=False).round(1))

# 3) Gráfico 1: cuántas desviaciones sobre lo habitual tuvo cada día (los críticos pasan de 2)
MESES_ABR = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep"]
fig, ax = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True)
for a, c, titulo in zip(ax, ["siniestros", "fallecidos"], ["Siniestros por día", "Fallecidos por día"]):
    z = d[f"z_{c}"]
    crit = d[z > 2]
    a.scatter(d.index[z <= 2], z[z <= 2], color=GRIS, s=14, alpha=0.6, label="Día normal")
    a.scatter(crit.index, z[z > 2], color="#e34948", s=34, zorder=3, label=f"Día crítico ({len(crit)})")
    # Rótulos en una fila arriba, separados lo justo para no montarse, con una línea guía hacia cada punto
    fechas = list(crit.index)
    xs = [f.toordinal() for f in fechas]
    sep, tope = 12, d.index[-1].toordinal() + 2
    for i in range(1, len(xs)):
        xs[i] = max(xs[i], xs[i - 1] + sep)
    xs[-1] = min(xs[-1], tope)
    for i in range(len(xs) - 2, -1, -1):
        xs[i] = min(xs[i], xs[i + 1] - sep)
    for fecha, x in zip(fechas, xs):
        a.annotate(f"{fecha.day} {MESES_ABR[fecha.month - 1]}", (fecha, z[fecha]), xytext=(pd.Timestamp.fromordinal(x), 6.4),
                   textcoords="data", ha="center", va="bottom", fontsize=8,
                   arrowprops=dict(arrowstyle="-", color="#9a9994", lw=0.7, shrinkA=1, shrinkB=3))
    a.axhline(2, color="#52514e", ls="--", lw=1)
    a.text(d.index[10], 2.1, "Límite: 2 desviaciones", ha="left", va="bottom", fontsize=8, color="#52514e")
    a.axhline(0, color="#c9c8c3", lw=0.8)
    a.set_ylim(-3, 7.6)
    a.set_ylabel("Desviaciones sobre lo habitual\n(0 = día normal)")
    a.set_title(titulo, loc="left")
    a.legend(frameon=False, loc="lower right", bbox_to_anchor=(1, 1.0), ncol=2)
ax[1].set_xlabel("Fecha (2026)")
guardar_fig(fig, "p02_dias_criticos_z")

# 4) Gráfico 2: ¿en qué se parecen los días críticos? Cada grupo de barras compara los días críticos con TODOS los días.
seccion("3) Qué tienen en común los días críticos (% de los críticos vs % de todos los días)")
causa4 = {"Distracción al conducir": "Distracción", "Exceso de velocidad": "Velocidad",
          "Irrespeto de señales / prioridad": "Señales"}
d["causa_dominante"] = sin.groupby("fecha")["causa_grupo"].agg(lambda s: s.value_counts().index[0]).reindex(d.index)
d["causa4"] = d["causa_dominante"].map(causa4).fillna("Otras")
d["mes"] = d.index.month
d["tipo_dia"] = d["feriado"].map({True: "Feriado", False: "No feriado"})
paneles = [("Día de la semana", d["dow"].map(lambda i: DIAS[i][:3].title()), [x[:3].title() for x in DIAS]),
           ("Mes", d["mes"].map(lambda m: MESES_ABR[m - 1].title()), [m.title() for m in MESES_ABR]),
           ("Feriado", d["tipo_dia"], ["Feriado", "No feriado"]),
           ("Causa dominante del día", d["causa4"], ["Distracción", "Velocidad", "Señales", "Otras"])]
fig, ax = plt.subplots(2, 2, figsize=(14, 7.5))
for a, (titulo, serie, orden) in zip(ax.ravel(), paneles):
    todos = 100 * serie.value_counts(normalize=True).reindex(orden).fillna(0)
    crit_ = 100 * serie[d["critico"]].value_counts(normalize=True).reindex(orden).fillna(0)
    print(f"\n{titulo}:")
    mostrar(pd.DataFrame({"% críticos": crit_, "% todos los días": todos}).round(1))
    x = range(len(orden))
    a.bar([i - 0.2 for i in x], todos, 0.4, color=GRIS, label=f"Todos los días ({len(d)})")
    b = a.bar([i + 0.2 for i in x], crit_, 0.4, color=PALETA[1], label=f"Días críticos ({len(cr)})")
    a.bar_label(b, fmt="%.0f", fontsize=8, padding=2)
    a.set_xticks(list(x))
    a.set_xticklabels(orden)
    a.set_ylabel("% de los días del grupo")
    a.set_title(titulo)
ax[1, 0].legend(frameon=False, loc="upper left")
guardar_fig(fig, "p02_que_tienen_en_comun")

### Respuesta a la pregunta 2

**Resultados**

- **13 días críticos de 273 (4.8 %)**: 6 por siniestros, 10 por fallecidos y 3 por ambos (1 de enero, 15 y 21 de junio). Correlación diaria entre siniestros y fallecidos: **0.64**.
- Peor día: **1 de enero** (123 siniestros y 21 fallecidos; 4.2 y 3.7 desviaciones). Le siguen el 15 de junio, el 21 de junio y el 5 de septiembre.
- Comparación de los 13 días críticos con los 273 días:

| Característica | Días críticos | Todos los días |
|---|---|---|
| Entre junio y septiembre | **85 %** (11 de 13) | 45 % |
| Feriado | **23 %** (3 de 13) | 6.6 % (18 de 273) |
| Causa dominante: distracción | **77 %** | 57.5 % |
| Causa dominante: velocidad | 23 % | 19.4 % |
| Causa dominante: irrespeto de señales | **0 %** | 18.7 % |
| Lunes o viernes | 46 % (6 de 13) | 29 % |

**Análisis**

1. **Qué días:** son pocos (4.8 %) y extremos; el 1 de enero es el más marcado en ambos criterios.
2. **Qué tienen en común:** caen sobre todo entre junio y septiembre, hay más feriados de los esperados y la causa dominante es más a menudo la distracción, mientras que el irrespeto de señales casi no aparece. El **día de la semana no los distingue** (se reparten entre los siete días).
3. **Siniestros y fallecidos no siempre suben juntos:** el 4 de agosto tuvo 50 siniestros (en el promedio) pero 12 fallecidos (3.2 desviaciones).

**Limitaciones**

- Con 13 casos, los porcentajes son frágiles; el feriado son solo 3 días.
- **El nivel diario sube durante el año** (de 52 siniestros por día en enero a 72 en agosto) y «habitual» se calcula con el promedio de todo el período, así que los meses de más actividad quedan por encima casi por construcción.
- La distracción es la causa dominante en 57.5 % de **todos** los días, por eso se compara contra ese total y no contra cero.

---
## Pregunta 3 · Dónde: Provincias y cantones

> **¿Qué provincias y cantones concentran la mayor parte de los siniestros y de las muertes, y el ranking cambia si se mide por volumen o por gravedad?**

**Técnicas:** clasificación Pareto ABC (80-15-5), concentración del Top 5, ranking por volumen y por gravedad, gráfico Pareto facetado y mapa de burbujas con latitud y longitud (Folium).

**Qué hace el código**

- `pareto_abc()` (en `utilidades.py`) ordena provincias y cantones, calcula el acumulado y asigna clase A (hasta 80 %), B (hasta 95 %) y C, para siniestros y para fallecidos.
- `gravedad()` da siniestros, fallecidos y % con fallecidos por provincia, con sus rangos por volumen y por gravedad (la tabla impresa filtra provincias con 100 o más siniestros).
- Grafica dos Pareto lado a lado y un mapa de burbujas de Folium (tamaño = volumen, color = % con fallecidos; rotula las 8 provincias con más siniestros y Esmeraldas, Zamora Chinchipe y Cañar). El mapa interactivo es el HTML y el PNG es su captura.

In [ ]:
import folium
import pandas as pd
import matplotlib.pyplot as plt
from branca.colormap import LinearColormap

from limpieza_datos import cargar_datos
from utilidades import (PALETA, etiqueta_mapa, gravedad, guardar_fig, guardar_mapa, guardar_tabla, mapa_base, mostrar,
                        pareto_abc, seccion)

sin, _ = cargar_datos()

# 1) Pareto ABC por provincia y cantón, midiendo siniestros y fallecidos
seccion("1) Clasificación ABC (80-15-5)")
res = {}
for nivel in ["provincia", "canton"]:
    for medida, serie in [("siniestros", sin.groupby(nivel).size()), ("fallecidos", sin.groupby(nivel)["fallecidos"].sum())]:
        abc = pareto_abc(serie)
        res[(nivel, medida)] = abc
        n_a = (abc["clase"] == "A").sum()
        print(f"{nivel:9s} | {medida:10s} | {len(abc)} unidades | clase A: {n_a} ({100 * n_a / len(abc):.0f}% de las unidades) "
              f"= 80% | Top 5 = {abc['%'].head(5).sum():.1f}%")
    guardar_tabla(res[(nivel, "siniestros")], f"p03_abc_{nivel}_siniestros")
print("\nTop 10 provincias por siniestros:")
mostrar(res[("provincia", "siniestros")].round(1), 10)

# 2) Volumen vs gravedad por provincia (% de siniestros con fallecidos)
seccion("2) Volumen vs gravedad por provincia (provincias con 100 o más siniestros)")
prov = gravedad(sin, "provincia")
prov[["lat", "lon"]] = sin.groupby("provincia")[["lat", "lon"]].mean()
prov["rango por volumen"] = prov["siniestros"].rank(ascending=False)
prov["rango por gravedad"] = prov["% fatales"].rank(ascending=False)
mostrar(prov[prov["siniestros"] >= 100].sort_values("% fatales", ascending=False)[
    ["siniestros", "fallecidos", "% fatales", "rango por volumen", "rango por gravedad"]].round(1), 12)
guardar_tabla(prov, "p03_provincias")

# 3) Gráficos: Pareto facetado (siniestros | fallecidos) y mapa de burbujas
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for a, medida, c in zip(ax, ["siniestros", "fallecidos"], PALETA):
    abc = res[("provincia", medida)].head(12)
    a.bar(range(len(abc)), abc["%"], color=c)
    a.plot(range(len(abc)), abc["% acumulado"], color="#52514e", marker="o", ms=4)
    a.axhline(80, color=PALETA[3], ls="--", lw=1)
    a.set_ylim(0, 105)
    a.set_xticks(range(len(abc)))
    a.set_xticklabels([("Santo Domingo" if i.startswith("SANTO DOMINGO") else i.title()[:14]) for i in abc.index],
                      rotation=60, ha="right")
    a.set_ylabel(f"% del total de {medida} (barras) y acumulado (línea)")
    a.set_title(f"Pareto de provincias: {medida}")
guardar_fig(fig, "p03_pareto_provincias")

# Mapa de burbujas con Folium: una burbuja por provincia en su coordenada promedio
mapa = mapa_base(location=[-1.4, -78.6], zoom_start=7, control_scale=True)
escala = LinearColormap(["#cde2fb", "#6da7ec", "#256abf", "#0d366b"], vmin=prov["% fatales"].min(),
                        vmax=prov["% fatales"].max(), caption="% de siniestros con fallecidos")
# Rótulos permanentes: las 8 provincias con más siniestros y las 3 más graves que cita el guion
rotular = set(prov.nlargest(8, "siniestros").index) | {"ESMERALDAS", "ZAMORA CHINCHIPE", "CAÑAR"}
LADO = {"GUAYAS": "bottom", "SANTA ELENA": "left", "SANTO DOMINGO DE LOS TSACHILAS": "left", "PICHINCHA": "right",
        "CAÑAR": "left", "ZAMORA CHINCHIPE": "bottom", "LOJA": "left", "AZUAY": "right"}
for n, r in prov.iterrows():
    nombre = "Santo Domingo" if n.startswith("SANTO DOMINGO") else n.title()
    radio = max(3.0, r["siniestros"] ** 0.5 / 3.5)
    marca = folium.CircleMarker([r["lat"], r["lon"]], radius=radio, color="white", weight=1,
                                fill=True, fill_color=escala(r["% fatales"]), fill_opacity=0.85,
                                popup=folium.Popup(f"<b>{nombre}</b><br>Siniestros: {int(r['siniestros']):,}<br>"
                                                   f"Fallecidos: {int(r['fallecidos']):,}<br>"
                                                   f"% con fallecidos: {r['% fatales']:.1f}", max_width=220))
    if n in rotular:
        lado = LADO.get(n, "top")                  # el rótulo se corre un radio para no tapar la burbuja
        dx, dy = {"right": (radio + 4, 0), "left": (-radio - 4, 0), "top": (0, -radio - 4), "bottom": (0, radio + 4)}[lado]
        marca.add_child(folium.Tooltip(nombre, permanent=True, direction=lado, offset=(dx, dy),
                                       style="font-size:11px;padding:1px 4px;"))
    marca.add_to(mapa)
escala.add_to(mapa)
etiqueta_mapa(mapa, "<b>Siniestros por provincia</b><br>tamaño = volumen, color = % con fallecidos", abajo=True)
cont = sin[sin["lon"] > -85]                       # el encuadre es el continente; Galápagos (5 siniestros) queda fuera de vista
mapa.fit_bounds([[cont["lat"].min(), cont["lon"].min()], [cont["lat"].max(), cont["lon"].max()]])
guardar_mapa(mapa, "p03_mapa_burbujas_provincias", ancho=800, alto=900)

### Respuesta a la pregunta 3

**Resultados**

| Medida | Unidades | Clase A (80 %) | Top 5 |
|---|---|---|---|
| Provincia, siniestros | 24 | 8 (33 %) | 70.3 % |
| Provincia, fallecidos | 24 | 10 (42 %) | 65.5 % |
| Cantón, siniestros | 210 | 35 (17 %) | 51.2 % |
| Cantón, fallecidos | 210 | 52 (25 %) | 39.1 % |

- Volumen: Guayas 5 637 (32.7 %) y Pichincha 3 507 (20.3 %) suman 53 %.
- Gravedad (100 o más siniestros): **Esmeraldas 25.3 %** (16.ª en volumen), **Zamora Chinchipe 23.3 %**, **Cañar 19.5 %**. Promedio nacional: 9.6 %.

**Análisis**

1. **El volumen está muy concentrado** y las muertes se reparten más (10 provincias y 52 cantones para el 80 %, frente a 8 y 35 de siniestros).
2. **El ranking cambia con la medida:** por volumen mandan Guayas y Pichincha; por gravedad suben provincias pequeñas.

**Limitaciones**

- **Santa Elena** pasa de 12 a 15 siniestros mensuales a ~75 desde marzo, todos del ente CTE: parece un cambio de cobertura y puede inflar su posición (7.º en volumen).
- Los porcentajes de provincias con pocos siniestros son inestables (de ahí el filtro de 100).
- Galápagos (5 siniestros) queda fuera del encuadre. Folium necesita internet para el mapa base y el PNG se captura con Edge o Chrome (si no hay, se captura a mano desde el HTML).

---
## Pregunta 4 · Dónde: Vehículos por provincia

> **¿En qué provincias pesan más las motocicletas, los camiones y los buses en los siniestros, y esas provincias son las mismas donde hay más siniestros?**

**Técnicas:** porcentaje de siniestros que involucran cada tipo de vehículo por provincia, ranking por porcentaje y por cantidad, y mapas de burbujas con latitud y longitud, uno por vehículo (Folium).

**Qué hace el código**

- Para cada provincia cuenta los siniestros totales y cuántos involucran al menos una motocicleta, un camión o un bus (`n_motocicleta`, `n_camion`, `n_bus`), y calcula el **porcentaje de los siniestros de la provincia** que incluyen ese vehículo. Solo usa las 20 provincias con 100 o más siniestros (quedan fuera Galápagos, Pastaza, Orellana y Napo, cuyo porcentaje es inestable). Guarda `p04_vehiculos_por_provincia.csv`.
- Dibuja tres mapas de burbujas de Folium lado a lado, uno por vehículo, con una burbuja por provincia en su coordenada promedio: el tamaño es el número de siniestros con ese vehículo y el color es el porcentaje de los siniestros de la provincia. Rotula las 3 provincias de mayor porcentaje y las 2 de mayor cantidad. En el HTML se puede hacer zoom y, al hacer clic en una burbuja, aparecen sus cifras.

In [ ]:
import folium
import numpy as np
import pandas as pd
from branca.colormap import LinearColormap

from limpieza_datos import cargar_datos
from utilidades import etiqueta_mapa, guardar_facetas, guardar_tabla, mapa_base, mostrar, seccion

sin, _ = cargar_datos()
MINIMO = 100                                   # provincias con menos siniestros se omiten: su porcentaje es inestable
VEHICULOS = {"Motocicleta": "n_motocicleta", "Camión": "n_camion", "Bus": "n_bus"}

# 1) Por provincia: siniestros totales y siniestros que involucran cada vehículo
seccion("1) Porcentaje de siniestros que involucran cada vehículo, por provincia")
prov = sin.groupby("provincia").agg(siniestros=("sid", "size"), lat=("lat", "mean"), lon=("lon", "mean"))
nacional = {}
for nombre, col in VEHICULOS.items():
    con = sin[sin[col] > 0]
    prov[f"{nombre} (n)"] = con.groupby("provincia").size()
    prov[f"{nombre} (n)"] = prov[f"{nombre} (n)"].fillna(0).astype(int)
    prov[f"{nombre} (%)"] = 100 * prov[f"{nombre} (n)"] / prov["siniestros"]
    nacional[nombre] = 100 * len(con) / len(sin)
    print(f"{nombre}: {len(con):,} siniestros ({nacional[nombre]:.1f}% del total)")
prov = prov[prov["siniestros"] >= MINIMO]
print(f"\nProvincias con {MINIMO} o más siniestros: {len(prov)} de 24")
guardar_tabla(prov, "p04_vehiculos_por_provincia")

for nombre in VEHICULOS:
    print(f"\n{nombre}: provincias con mayor porcentaje (promedio nacional {nacional[nombre]:.1f}%)")
    mostrar(prov.sort_values(f"{nombre} (%)", ascending=False)[["siniestros", f"{nombre} (n)", f"{nombre} (%)"]].round(1), 5)
    top2 = prov[f"{nombre} (n)"].nlargest(2)
    print(f"Mayor cantidad: {', '.join(top2.index.str.title())} "
          f"({100 * top2.sum() / sin[sin[VEHICULOS[nombre]] > 0].shape[0]:.1f}% de los siniestros con {nombre.lower()})")

# 2) Un mapa por vehículo (facetas): tamaño = siniestros con ese vehículo, color = % de los siniestros de la provincia
seccion("2) Mapas de burbujas por vehículo")
LADO = {"GUAYAS": "bottom", "SANTA ELENA": "left", "SANTO DOMINGO DE LOS TSACHILAS": "left", "PICHINCHA": "right",
        "CAÑAR": "right", "LOS RIOS": "left", "MANABI": "left", "EL ORO": "left", "LOJA": "left", "COTOPAXI": "right",
        "TUNGURAHUA": "right", "CHIMBORAZO": "right", "AZUAY": "right", "ZAMORA CHINCHIPE": "bottom"}
cont = sin[sin["lon"] > -85]                    # encuadre del continente; Galápagos queda fuera de vista
facetas = []
for nombre in VEHICULOS:
    pct, cant = f"{nombre} (%)", f"{nombre} (n)"
    mapa = mapa_base(location=[-1.4, -78.6], zoom_start=7, zoom_snap=1)   # zoom entero: sin líneas entre teselas
    escala = LinearColormap(["#cde2fb", "#6da7ec", "#256abf", "#0d366b"], vmin=prov[pct].min(), vmax=prov[pct].max(),
                            caption=f"% de siniestros de la provincia con {nombre.lower()}")
    rotular = set(prov[pct].nlargest(3).index) | set(prov[cant].nlargest(2).index)    # las 3 de mayor porcentaje y las 2 de mayor cantidad
    for n, r in prov.iterrows():
        etiqueta = "Santo Domingo" if n.startswith("SANTO DOMINGO") else n.title()
        radio = max(4.0, r[cant] ** 0.5 / 2.2)
        marca = folium.CircleMarker([r["lat"], r["lon"]], radius=radio, color="white", weight=1, fill=True,
                                    fill_color=escala(r[pct]), fill_opacity=0.85,
                                    popup=folium.Popup(f"<b>{etiqueta}</b><br>Siniestros: {int(r['siniestros']):,}<br>"
                                                       f"Con {nombre.lower()}: {int(r[cant]):,} ({r[pct]:.1f} %)", max_width=220))
        if n in rotular:
            lado = LADO.get(n, "top")
            dx, dy = {"right": (radio + 4, 0), "left": (-radio - 4, 0), "top": (0, -radio - 4), "bottom": (0, radio + 4)}[lado]
            marca.add_child(folium.Tooltip(etiqueta, permanent=True, direction=lado, offset=(dx, dy),
                                           style="font-size:11px;padding:1px 4px;"))
        marca.add_to(mapa)
    escala.add_to(mapa)
    etiqueta_mapa(mapa, f"<b>{nombre}</b> (promedio nacional {nacional[nombre]:.0f} %)<br>tamaño = siniestros con {nombre.lower()}", abajo=True)
    mapa.fit_bounds([[cont["lat"].min(), cont["lon"].min()], [cont["lat"].max(), cont["lon"].max()]])
    facetas.append(mapa)
guardar_facetas(facetas, "p04_vehiculos_por_provincia", ancho=1800, alto=900)

### Respuesta a la pregunta 4

**Resultados**

Siniestros que involucran cada vehículo (de 17 236): motocicleta 5 647 (**32.8 %**), camión 1 680 (**9.7 %**), bus 862 (**5.0 %**).

| Vehículo | Provincias con mayor porcentaje | Provincias con menor porcentaje | Mayor cantidad |
|---|---|---|---|
| Motocicleta (33 % nacional) | Santo Domingo **50.4 %**, Los Ríos 46.5 %, Esmeraldas 40.4 % | Zamora Chinchipe 17.8 %, Tungurahua 19.6 % | Guayas 1 934, Pichincha 1 015 |
| Camión (10 % nacional) | Cotopaxi **17.4 %**, Cañar 16.6 %, Carchi 16.1 %, Los Ríos 15.8 % | Santa Elena 5.8 %, Pichincha 6.7 % | Guayas 572, Pichincha 235 |
| Bus (5 % nacional) | Chimborazo **8.4 %**, Imbabura 8.3 %, Pichincha 7.2 % | El Oro 2.1 %, Bolívar 3.0 % | Pichincha 254, Guayas 223 |

Guayas y Pichincha concentran el 52.2 % de los siniestros con moto, el 48.0 % de los de camión y el 55.3 % de los de bus. Dentro de sus provincias el porcentaje es: Guayas 34.3 % con moto, 10.1 % con camión y 4.0 % con bus; Pichincha, 28.9 %, 6.7 % y 7.2 %.

**Análisis**

1. **Las provincias que destacan cambian según el vehículo:** las motos en Santo Domingo, Los Ríos y Esmeraldas (más del 40 %); los camiones en Cotopaxi, Cañar, Carchi y Los Ríos; los buses en Chimborazo, Imbabura y Pichincha.
2. **Porcentaje y cantidad cuentan historias distintas.** Por cantidad mandan Guayas y Pichincha en los tres vehículos, sobre todo porque tienen más siniestros en total (53.1 % del país); por porcentaje destacan provincias más pequeñas.
3. **Las diferencias son grandes:** la moto pasa de 17.8 % (Zamora Chinchipe) a 50.4 % (Santo Domingo); el camión, de 5.8 % (Santa Elena) a 17.4 % (Cotopaxi), tres veces más; el bus, de 2.1 % (El Oro) a 8.4 % (Chimborazo).

**Limitaciones**

- **Muestra dónde participa cada vehículo, no cuál es más peligroso.** La gravedad por vehículo se trata en la pregunta 7.
- **No hay datos del parque automotor ni de kilómetros recorridos:** un porcentaje alto puede reflejar que en esa provincia circulan más vehículos de ese tipo, y no que sean más riesgosos.
- Un siniestro puede involucrar varios vehículos, por eso los porcentajes de moto, camión y bus no suman 100 %. "Camión" es solo esa categoría; no incluye camioneta ni furgoneta.
- La escala de color es distinta en cada mapa (moto de 18 a 50 %, camión de 6 a 17 %, bus de 2 a 8 %): sirve para comparar provincias dentro de un mismo vehículo, no entre vehículos.
- Los porcentajes de provincias con pocos casos (Zamora Chinchipe, 129 siniestros; Carchi, 118) son más variables.
- Folium necesita internet para el mapa base y el PNG se captura con Edge o Chrome (si no hay, se captura a mano desde el HTML).

---
## Pregunta 5 · Dónde: Zona, jerarquía vial y corredores

> **¿Cómo influyen la zona (urbana o rural), la jerarquía de la vía y los corredores viales en la cantidad y gravedad de los siniestros, y cómo cambia eso a lo largo del día?**

**Técnicas:** comparación urbano / rural, ranking de corredores viales, barras agrupadas por franja horaria y zona, gráfico de burbujas y heatmap corredor x franja.

**Qué hace el código**

- `gravedad()` por zona y por red vial: compara el % de fatales dentro y fuera de la red estatal (un tercio de los siniestros; el resto tiene "no aplica" en los campos viales) y por jerarquía y corredor (mínimo 60 siniestros).
- Un gráfico de barras agrupadas con el % de fatales por franja horaria, con la zona urbana (azul) junto a la rural (naranja) en cada franja, y a su derecha los 12 corredores principales en burbujas (X = siniestros, Y = % fatales, tamaño = fallecidos, línea punteada = promedio de la red estatal).
- Un heatmap corredor × franja con los 10 corredores de más siniestros.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from limpieza_datos import FRANJAS, cargar_datos
from utilidades import CMAP_SEC, PALETA, gravedad, guardar_fig, guardar_tabla, mostrar, seccion

sin, _ = cargar_datos()

# 1) Urbano vs rural
seccion("1) Zona urbana vs rural")
z = gravedad(sin, "zona")
z["hora mediana"] = sin.groupby("zona")["hora_decimal"].median()
mostrar(z.round(2))
guardar_tabla(z, "p05_zona")

# 2) Red vial estatal: jerarquía y corredores (E35, E25, ...)
seccion("2) Red vial estatal")
via = sin[sin["via_estatal"]]
print(f"Siniestros en la red vial estatal: {len(via):,} ({100 * len(via) / len(sin):.1f}%); "
      f"% fatales {100 * via['fatal'].mean():.1f}% vs {100 * sin.loc[~sin['via_estatal'], 'fatal'].mean():.1f}% fuera de ella")
mostrar(gravedad(via.dropna(subset=["jerarquia_via"]), "jerarquia_via").round(2))
corr = gravedad(via, "ubicacion_via", minimo=60).sort_values("siniestros", ascending=False)
mostrar(corr.round(1), 15)
guardar_tabla(corr, "p05_corredores")

# 3) Barras agrupadas: gravedad por franja horaria, con la zona urbana junto a la rural, y burbujas de corredores
seccion("3) Gravedad por franja horaria y zona")
f = gravedad(sin, ["zona", "franja"]).reset_index()
mostrar(f.round(1))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.2]})
for desp, zona, c in [(-0.2, "URBANA", PALETA[0]), (0.2, "RURAL", PALETA[1])]:
    d = f[f["zona"] == zona].set_index("franja").reindex(FRANJAS)
    barras = ax[0].bar([i + desp for i in range(4)], d["% fatales"], 0.4, color=c,
                       label=f"Zona {zona.lower()} (n = {int(d['siniestros'].sum()):,})")
    ax[0].bar_label(barras, fmt="%.1f", fontsize=8, padding=2)
ax[0].set_xticks(range(4))
ax[0].set_xticklabels([x.split(" ")[0] for x in FRANJAS])
ax[0].set_ylim(0, 24)
ax[0].set_xlabel("Franja horaria")
ax[0].set_ylabel("% de siniestros con fallecidos")
ax[0].set_title("Gravedad por franja horaria: urbana vs rural")
ax[0].legend(frameon=False, loc="upper right")
# Burbujas: corredores viales
top = corr.head(12)
ax[1].scatter(top["siniestros"], top["% fatales"], s=top["fallecidos"] * 4, color=PALETA[0], alpha=0.75, edgecolor="white")
for n_, r in top.iterrows():
    ax[1].annotate(n_, (r["siniestros"], r["% fatales"]), fontsize=8, xytext=(5, 4), textcoords="offset points")
ax[1].axhline(100 * via["fatal"].mean(), color="#52514e", ls="--", lw=1, label="Promedio de la red estatal")
ax[1].legend(frameon=False, loc="upper right")
ax[1].set_xlabel("Siniestros en el corredor")
ax[1].set_ylabel("% de siniestros con fallecidos")
ax[1].set_title("Corredores viales (tamaño = fallecidos)")
guardar_fig(fig, "p05_zona_franja_corredores")

# 4) Heatmap corredor x franja (siniestros) para los 10 corredores principales
m = via[via["ubicacion_via"].isin(top.index[:10])].pivot_table(index="ubicacion_via", columns="franja", values="sid",
                                                              aggfunc="count", observed=False).reindex(top.index[:10])
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(m, annot=True, fmt="d", cmap=CMAP_SEC, ax=ax, cbar_kws={"label": "Siniestros"})
ax.set_xlabel("Franja horaria")
ax.set_ylabel("Corredor vial (código de la red estatal)")
ax.set_xticklabels([x.get_text().split(" ")[0] for x in ax.get_xticklabels()], rotation=0)
guardar_fig(fig, "p05_heatmap_corredor_franja")

### Respuesta a la pregunta 5

**Resultados**

| | Urbana | Rural |
|---|---|---|
| Siniestros | 10 939 | 6 297 |
| % con fallecidos | 7.09 | **13.97** |
| Madrugada (00-05) | 10.7 | **20.2** |
| Mañana / Tarde / Noche | 5.8 / 5.1 / 7.7 | 12.2 / 11.9 / 14.0 |

- Red estatal: **13.7 %** de fatales frente a 7.5 % fuera de ella. Arterial 13.9 % y colectora 13.5 %: casi iguales.
- Corredores (siniestros, % fatales): E40 896, 10.5 · E35 722, 12.3 · **E25** 645, **17.1** (125 fallecidos, el que más) · **E20** 210, **22.4** · E482 107, 19.6.

**Análisis**

1. **La zona rural es el doble de grave** aunque tiene menos siniestros. La red estatal también, pero no es evidencia independiente: el 69 % de sus siniestros son rurales (dentro de la red, rural 14.7 % y urbana 11.6 %).
2. **La madrugada es la franja más letal** en ambas zonas, pese a tener el menor volumen (de madrugada pesan más velocidad, alcohol y fatiga, pregunta 6).
3. **Volumen y gravedad no coinciden en los corredores:** E40 tiene más siniestros pero menos letalidad que E25, E20 y E482.

**Limitaciones**

- Solo hay datos de vía para un tercio de los siniestros; los corredores no describen la red urbana ni municipal.
- Otros corredores también superan el promedio estatal (E45 17.2 % con 198 siniestros, E48 15.0 % con 253); se destacan E25, E20 y E482 por combinar letalidad y volumen. E482 y E20 tienen pocos casos y sus porcentajes son variables.

---
## Pregunta 6 · Por qué: Causas y factores

> **¿Qué causas probables (y qué factores: humano, vehículo o vía) explican la mayor parte de los siniestros, cuáles se asocian a mayor gravedad y cambia su peso según la hora del día?**

**Técnicas:** Pareto de causas, porcentaje por factor, gravedad por grupo de causa, gráfico de burbujas (frecuencia vs gravedad) y heatmaps causa x franja horaria.

**Qué hace el código**

- Cuenta los siniestros por causa (26 códigos), aplica `pareto_abc()` y calcula el % por factor causal (humano, vehículo, vía y entorno).
- `gravedad()` por cada uno de los 10 grupos de causa: siniestros, fallecidos y % con fallecidos.
- Grafica el Top 10 de causas en barras, coloreadas por factor causal y con una leyenda que da el % de todos los siniestros de cada factor (humano 94.5 %, vehículo 1.2 %, vía y entorno 4.3 %), y un gráfico de burbujas (X = frecuencia en escala log, Y = % fatales, tamaño = fallecidos, color = factor, línea = promedio nacional 9.6 %). Dos heatmaps causa × franja: peso de cada causa y % fatales (celdas con 30 o más siniestros).

In [ ]:
import textwrap

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

from limpieza_datos import cargar_datos
from utilidades import CMAP_SEC, PALETA, gravedad, guardar_fig, guardar_tabla, mostrar, pareto_abc, seccion

sin, _ = cargar_datos()

# 1) Pareto de causas probables (26 códigos) y factor contribuyente
seccion("1) Frecuencia de causas")
corto = sin.groupby("cod_causa")["causa_probable"].agg(
    lambda s: textwrap.shorten(s.iloc[0].split("(")[0].strip().rstrip(".").capitalize(), 58, placeholder="…"))
abc = pareto_abc(sin.groupby("cod_causa").size())
abc["causa"] = abc.index.map(corto)
mostrar(abc[["causa", "valor", "%", "% acumulado", "clase"]].round(1), 12)
print("\nSiniestros por factor causal (modelo humano - vehículo - vía):")
print((sin["factor_causal"].value_counts(normalize=True) * 100).round(1).to_string())

# 2) Gravedad por grupo de causa
seccion("2) Gravedad por grupo de causa")
g = gravedad(sin, "causa_grupo").sort_values("% fatales", ascending=False)
mostrar(g.round(2), 12)
guardar_tabla(g, "p06_gravedad_por_grupo_causa")

# 3) Gráficos: Pareto horizontal y burbujas frecuencia vs gravedad
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1, 1.15]})
col = {"Humano": PALETA[0], "Vehículo": PALETA[2], "Vía y entorno": PALETA[3]}
pct_factor = sin["factor_causal"].value_counts(normalize=True) * 100     # % de todos los siniestros por factor
factor_cod = sin.groupby("cod_causa")["factor_causal"].first()
top = abc.head(10).iloc[::-1]
ax[0].barh(top["causa"], top["%"], color=[col[factor_cod[c]] for c in top.index])
for y, v_ in enumerate(top["%"]):
    ax[0].text(v_ + 0.3, y, f"{v_:.1f}%", va="center", fontsize=8)
ax[0].set_xlabel("% de siniestros")
ax[0].set_title("Top 10 causas probables (color = factor causal)")
ax[0].legend(handles=[Patch(color=c, label=f"{k}: {pct_factor[k]:.1f} % de todos los siniestros") for k, c in col.items()],
             frameon=False, loc="lower right", fontsize=9)
fac =sin.groupby("causa_grupo")["factor_causal"].first()
OFF = {"Vehículo / carga": (6, -13), "Distracción al conducir": (-25, 30), "Exceso de velocidad": (-20, -38),
       "Vía y entorno": (-88, 6)}
for gname, r in g.iterrows():
    ax[1].scatter(r["siniestros"], r["% fatales"], s=r["fallecidos"] * 3 + 20, color=col[fac[gname]], alpha=0.8,
                  edgecolor="white")
    ax[1].annotate(gname, (r["siniestros"], r["% fatales"]), fontsize=8, textcoords="offset points",
                   xytext=OFF.get(gname, (6, 4)),
                   arrowprops=dict(arrowstyle="-", color="#8a8a85", lw=0.6) if gname in OFF else None)
ax[1].axhline(100 * sin["fatal"].mean(), color="#52514e", ls="--", lw=1, label="Promedio nacional")
ax[1].set_xscale("log")
ax[1].set_xlabel("Siniestros (escala logarítmica)")
ax[1].set_ylabel("% de siniestros con fallecidos")
ax[1].set_title("Frecuencia vs gravedad (tamaño = fallecidos)")
for k, c in col.items():
    ax[1].scatter([], [], color=c, label=k)
ax[1].legend(title="Factor", frameon=False, loc="lower left")
guardar_fig(fig, "p06_pareto_y_burbujas")

# 4) Heatmaps: peso de cada causa por franja horaria y % fatal (celdas con menos de 30 siniestros omitidas)
n = pd.crosstab(sin["causa_grupo"], sin["franja"])
share = n.div(n.sum(axis=0), axis=1) * 100
fat = sin.pivot_table(index="causa_grupo", columns="franja", values="fatal", aggfunc="mean", observed=False) * 100
fat = fat.astype(float).where(n >= 30)
orden = g.index
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
sns.heatmap(share.loc[orden], annot=True, fmt=".0f", cmap=CMAP_SEC, ax=ax[0], cbar_kws={"label": "% de los siniestros de la franja"})
ax[0].set_title("Peso de cada causa dentro de la franja horaria")
sns.heatmap(fat.loc[orden], annot=True, fmt=".0f", cmap=CMAP_SEC, ax=ax[1], cbar_kws={"label": "% de siniestros con fallecidos"})
ax[1].set_title("Gravedad por causa y franja (n ≥ 30)")
for a in ax:
    a.set_xlabel("Franja horaria")
    a.set_ylabel("")
    a.set_xticklabels([t.get_text().split(" ")[0] for t in a.get_xticklabels()], rotation=0)
guardar_fig(fig, "p06_heatmaps_causa_franja")

### Respuesta a la pregunta 6

**Resultados**

- Factor causal: **humano 94.5 %**, vía y entorno 4.3 %, vehículo 1.2 %.
- Causas más frecuentes: distracción **28.8 %**, velocidad **19.3 %**, no respetar señales 13.5 %, distancia prudencial 6.4 %, alcohol 5.6 %.

| Grupo (promedio nacional 9.6 %) | Siniestros | % con fallecidos |
|---|---|---|
| Vía y entorno | 744 | 12.8 |
| Distracción al conducir | 4 956 | 12.2 |
| Exceso de velocidad | 3 331 | 12.0 |
| Alcohol / sustancias | 1 044 | 5.8 |
| Irrespeto de señales / prioridad | 3 382 | 4.8 |

- De madrugada: velocidad 26.0 % (16.8 % en la tarde), alcohol 10.0 % (3.2 % en la mañana) y fatiga 2.7 %; el irrespeto de señales cae de 23.0 % a 12.2 %. La distracción se mantiene en 28 a 30 % a toda hora.

**Análisis**

1. **Casi todo es factor humano:** las causas registradas son casi todas del comportamiento del conductor, no del vehículo ni de la vía.
2. **Distracción y velocidad explican casi la mitad (48.1 %)** y están entre las más graves; frecuente no es lo mismo que grave (el irrespeto de señales es la tercera causa y de las menos letales).
3. **La madrugada cambia el peso de las causas** justo cuando sube la letalidad (pregunta 5).

**Limitaciones**

- **El alcohol sale poco letal (5.8 %), lo que contradice la intuición.** Probable sesgo de registro (la causa se anota en controles, más frecuentes en siniestros leves); es una hipótesis que el dataset no confirma.
- La causa es la *probable* que anota el ente de control, no una causa verificada.

---
## Pregunta 7 · Qué y a quién: Tipo de siniestro y vehículos

> **¿Qué tipos de siniestro y qué vehículos involucrados concentran la mayor gravedad, y esa gravedad cambia entre zona urbana y rural?**

**Técnicas:** tabla de gravedad por tipo de siniestro, comparación del % de siniestros fatales con y sin cada tipo de vehículo, barras agrupadas urbano vs rural y heatmap tipo de siniestro x vehículo.

**Qué hace el código**

- `gravedad()` por `tipo_siniestro` (13 tipos), ordenado por % con fallecidos.
- Para cada tipo de vehículo (30 o más siniestros) compara el % de fatales *cuando participa* contra *cuando no*, y calcula las "veces más fatales". También mira el % según el número de vehículos.
- Grafica en una sola figura dos paneles: a la izquierda, barras agrupadas urbana / rural por tipo (100 o más casos, con el número de siniestros entre paréntesis y la línea del promedio nacional); a la derecha, barras con y sin cada vehículo con las veces más fatales. Aparte, un heatmap tipo de siniestro × vehículo 1 (celdas con 30 o más casos).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from limpieza_datos import cargar_datos
from utilidades import CMAP_SEC, GRIS, PALETA, gravedad, guardar_fig, guardar_tabla, mostrar, seccion

sin, _ = cargar_datos()

# 1) Gravedad por tipo de siniestro
seccion("1) Gravedad por tipo de siniestro")
t = gravedad(sin, "tipo_siniestro").sort_values("% fatales", ascending=False)
mostrar(t.round(2))
guardar_tabla(t, "p07_gravedad_por_tipo_siniestro")

# 2) % de siniestros fatales cuando participa cada vehículo frente a cuando no participa
seccion("2) Gravedad según el vehículo involucrado")
veh = ["automovil", "motocicleta", "camioneta", "suv", "camion", "bus", "bicicleta", "scooter", "furgoneta"]
filas = []
for v in veh:
    con, sin_v = sin[sin[f"n_{v}"] > 0], sin[sin[f"n_{v}"] == 0]
    if len(con) < 30:
        continue
    filas.append({"vehiculo": v, "siniestros": len(con), "% fatales (con el vehículo)": 100 * con["fatal"].mean(),
                  "% fatales (sin él)": 100 * sin_v["fatal"].mean()})
rr = pd.DataFrame(filas).set_index("vehiculo")
rr["veces más fatales"] = rr["% fatales (con el vehículo)"] / rr["% fatales (sin él)"]
rr = rr.sort_values("veces más fatales", ascending=False)
mostrar(rr.round(2))
guardar_tabla(rr, "p07_gravedad_por_vehiculo")
print("\n% fatales según número de vehículos (4 = 4 o más):")
nv = sin.assign(nveh=sin["n_vehiculos"].clip(upper=4)).groupby("nveh")["fatal"].agg(["size", "mean"])
nv["mean"] *= 100
mostrar(nv.rename(columns={"size": "siniestros", "mean": "% fatales"}).round(2))

# 3) Gráfico: % de siniestros con fallecidos por tipo de siniestro (urbana vs rural) y por vehículo (con vs sin él)
seccion("3) Gravedad urbana vs rural por tipo de siniestro")
z = gravedad(sin, ["tipo_siniestro", "zona"], minimo=30).reset_index()
orden = list(t.index[t["siniestros"] >= 100])               # ya viene ordenado de mayor a menor gravedad
media = 100 * sin["fatal"].mean()
fig, ax = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.15, 1]})
alto = 0.38
for desp, zona, c in [(-alto / 2, "URBANA", PALETA[0]), (alto / 2, "RURAL", PALETA[1])]:
    d = z[z["zona"] == zona].set_index("tipo_siniestro").reindex(orden)
    b = ax[0].barh([i + desp for i in range(len(orden))], d["% fatales"], alto, color=c, label=f"Zona {zona.lower()}")
    ax[0].bar_label(b, fmt="%.0f", fontsize=8, padding=2)
ax[0].set_yticks(range(len(orden)))
ax[0].set_yticklabels([f"{i.capitalize()} ({int(t.loc[i, 'siniestros']):,})" for i in orden])
ax[0].invert_yaxis()
ax[0].axvline(media, color="#52514e", ls="--", lw=1)
ax[0].text(media + 0.6, len(orden) - 0.6, f"Promedio nacional ({media:.1f} %)", fontsize=8, color="#52514e")
ax[0].set_xlabel("% de siniestros con fallecidos")
ax[0].set_title("Qué tipo de siniestro: urbana vs rural")
ax[0].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
rv = rr.sort_values("veces más fatales")
b = ax[1].barh([i + alto / 2 for i in range(len(rv))], rv["% fatales (con el vehículo)"], alto, color=PALETA[0], label="Siniestros CON el vehículo")
ax[1].barh([i - alto / 2 for i in range(len(rv))], rv["% fatales (sin él)"], alto, color=GRIS, label="Siniestros SIN el vehículo")
for i, (_, r) in enumerate(rv.iterrows()):
    ax[1].text(r["% fatales (con el vehículo)"] + 0.3, i + alto / 2, f"{r['veces más fatales']:.2f}×", va="center", fontsize=8)
ax[1].set_yticks(range(len(rv)))
NOMBRES = {"camion": "Camión", "suv": "SUV", "automovil": "Automóvil"}
ax[1].set_yticklabels([f"{NOMBRES.get(i, i.capitalize())} ({int(n):,})" for i, n in rv["siniestros"].items()])
ax[1].axvline(media, color="#52514e", ls="--", lw=1)
ax[1].set_xlabel("% de siniestros con fallecidos (× = veces más fatales con el vehículo que sin él)")
ax[1].set_title("Qué vehículo: con vs sin él")
ax[1].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
guardar_fig(fig, "p07_gravedad_tipo_y_vehiculo")

# 4) Heatmap tipo de siniestro x vehículo 1 (% fatales; celdas con menos de 30 siniestros omitidas)
sub = sin[sin["tipo_veh_1"].notna()]
m = sub.pivot_table(index="tipo_siniestro", columns="tipo_veh_1", values="fatal", aggfunc="mean") * 100
n = sub.pivot_table(index="tipo_siniestro", columns="tipo_veh_1", values="fatal", aggfunc="size")
m = m.astype(float).where(n >= 30).dropna(how="all").dropna(axis=1, how="all")
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(m, annot=True, fmt=".0f", cmap=CMAP_SEC, ax=ax, cbar_kws={"label": "% de siniestros con fallecidos"})
ax.set_xlabel("Tipo de vehículo 1 (celdas con n < 30 omitidas)")
ax.set_ylabel("Tipo de siniestro")
ax.set_xticklabels([x.get_text().title().replace("Vehículo Deportivo Utilitario", "Utilitario deportivo") for x in ax.get_xticklabels()], rotation=45, ha="right")
ax.set_yticklabels([x.get_text().capitalize() for x in ax.get_yticklabels()], rotation=0)
guardar_fig(fig, "p07_heatmap_tipo_vehiculo")

### Respuesta a la pregunta 7

**Resultados**

| Tipo | Siniestros | % con fallecidos |
|---|---|---|
| Arrollamientos | 153 | **45.8** |
| Pérdida de pista | 504 | 22.4 |
| Choque frontal | 1 142 | 19.4 |
| Atropellos | 1 992 | 14.2 |
| Choque lateral | 4 858 | 4.8 |

- Veces más fatales con el vehículo: **bus 1.85**, **moto 1.72**, camión 1.44, automóvil 0.33.
- Por número de vehículos: 1 vehículo 12.3 % fatales, 2 vehículos 7.2 %.
- Urbano → rural: arrollamientos 37 → 59 %, atropellos 11 → 21 %, choque frontal 14 → 23 %, volcamientos 10 → 19 %. Solo «Otros» es más grave en la ciudad (9 % contra 7 %).

**Análisis**

1. **Arrollamientos, pérdidas de pista y choques frontales son los más graves;** lo más frecuente (choque lateral) es de los menos graves.
2. **Bus y moto elevan el riesgo;** es una asociación, no una causa.
3. **La zona rural es más grave en casi todos los tipos** (la excepción es «Otros»).

**Limitaciones**

- Arrollamientos tiene solo 153 casos y 74 de ellos con vehículo no identificado. Bicicleta (100) y scooter (90) tienen pocos casos.
- Las razones comparan con y sin vehículo sin controlar vía, velocidad ni hora; hay un 22 % de siniestros con el vehículo 1 sin identificar.

---
## Pregunta 8 · Qué y a quién: Perfil de las víctimas

> **¿Qué perfil (edad, sexo y rol en la vía) tienen las personas lesionadas y fallecidas, y qué grupos son proporcionalmente más vulnerables?**

**Técnicas:** tabla de participantes en formato largo, letalidad (fallecidos / víctimas) por perfil, edad mediana, pirámide de población facetada por condición, barras agrupadas sexo x edad y heatmap rol x grupo de edad.

**Qué hace el código**

- Usa la tabla de participantes (37 096 personas) y define víctimas = lesionados + fallecidos.
- Calcula la **letalidad = fallecidos / (fallecidos + lesionados)** por rol, sexo y grupo de edad, la edad mediana y la razón hombre : mujer. Compara también la letalidad con y sin edad registrada.
- Dibuja dos pirámides (lesionados y fallecidos) por edad y sexo y, en una segunda figura, barras de letalidad por sexo y edad junto a un heatmap de letalidad rol × edad (celdas con 30 o más personas).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from limpieza_datos import GRUPOS_EDAD, cargar_datos
from utilidades import CMAP_SEC, PALETA, guardar_fig, guardar_tabla, mostrar, seccion

_, part = cargar_datos()

# 0) Cobertura de datos: cuántas personas tienen edad / sexo / condición válidos
seccion("0) Cobertura de datos de las personas involucradas")
print(f"Personas registradas: {len(part):,}")
for c in ["edad", "sexo", "condicion"]:
    print(f"  {c}: {100 * part[c].notna().mean():.1f}% con dato válido")
vic = part[part["condicion"].isin(["LESIONADO", "FALLECIDO"])].copy()       # víctimas
vic["fallecido"] = vic["condicion"].eq("FALLECIDO")
print(f"Víctimas (lesionados + fallecidos): {len(vic):,} | con edad: {100 * vic['edad'].notna().mean():.1f}% "
      f"| con sexo: {100 * vic['sexo'].notna().mean():.1f}%")

# 1) Letalidad entre víctimas por rol, sexo y grupo de edad
seccion("1) Letalidad entre víctimas (fallecidos / (fallecidos + lesionados))")
tablas = {}
for var in ["rol", "sexo", "grupo_edad"]:
    d = vic.dropna(subset=[var]).groupby(var, observed=True).agg(victimas=("fallecido", "size"), fallecidos=("fallecido", "sum"))
    d["letalidad %"] = 100 * d["fallecidos"] / d["victimas"]
    tablas[var] = d
    print(f"\nPor {var}:")
    mostrar(d.round(2))
guardar_tabla(pd.concat(tablas), "p08_letalidad_por_perfil")
e = vic.dropna(subset=["edad"])
print(f"\nEdad mediana: fallecidos {e.loc[e['fallecido'], 'edad'].median():.0f} años | lesionados {e.loc[~e['fallecido'], 'edad'].median():.0f} años")
con_e, sin_e = vic[vic["edad"].notna()], vic[vic["edad"].isna()]
print(f"Letalidad entre víctimas con edad registrada: {100 * con_e['fallecido'].mean():.1f}% | sin edad: {100 * sin_e['fallecido'].mean():.1f}%")
hm = vic[vic["sexo"].isin(["HOMBRE", "MUJER"])]["sexo"].value_counts()
print(f"Razón hombre : mujer entre víctimas con sexo conocido = {hm['HOMBRE'] / hm['MUJER']:.1f} : 1")

# 2) Pirámide de población facetada: lesionados | fallecidos
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for a, cond in zip(ax, ["LESIONADO", "FALLECIDO"]):
    d = vic[(vic["condicion"] == cond) & vic["grupo_edad"].notna() & vic["sexo"].isin(["HOMBRE", "MUJER"])]
    c = pd.crosstab(d["grupo_edad"], d["sexo"]).reindex(GRUPOS_EDAD, fill_value=0)
    a.barh(range(len(c)), -c["HOMBRE"], color=PALETA[0], label="Hombres")
    a.barh(range(len(c)), c["MUJER"], color=PALETA[1], label="Mujeres")
    a.set_yticks(range(len(c)))
    a.set_yticklabels(c.index)
    mx = c.values.max() * 1.1
    a.set_xlim(-mx, mx)
    a.set_xticks(a.get_xticks())
    a.set_xticklabels([f"{abs(int(x)):,}" for x in a.get_xticks()])
    a.set_title(f"{cond.capitalize()}s")
    a.set_xlabel("Personas (con edad y sexo conocidos)")
    a.axvline(0, color="#52514e", lw=0.8)
ax[0].set_ylabel("Grupo de edad (años)")
ax[0].legend(frameon=False)
guardar_fig(fig, "p08_piramide_lesionados_fallecidos")

# 3) Vulnerabilidad: letalidad por edad y sexo (barras agrupadas) y por rol y edad (heatmap).
#    Se omiten las celdas con menos de 30 personas.
seccion("2) Letalidad por sexo y grupo de edad")
ls = vic[vic["sexo"].isin(["HOMBRE", "MUJER"])].dropna(subset=["grupo_edad"])
let_se = ls.pivot_table(index="sexo", columns="grupo_edad", values="fallecido", aggfunc="mean", observed=False)[GRUPOS_EDAD] * 100
n_se = ls.pivot_table(index="sexo", columns="grupo_edad", values="fallecido", aggfunc="size", observed=False)[GRUPOS_EDAD]
mostrar(pd.concat({"letalidad %": let_se, "víctimas": n_se}, axis=1).round(1))
guardar_tabla(pd.concat({"letalidad %": let_se, "víctimas": n_se}, axis=1), "p08_letalidad_sexo_edad")
d = vic.dropna(subset=["grupo_edad"])
let = d.pivot_table(index="rol", columns="grupo_edad", values="fallecido", aggfunc="mean", observed=False) * 100
n = d.pivot_table(index="rol", columns="grupo_edad", values="fallecido", aggfunc="size", observed=False)
let = let.astype(float).where(n >= 30).dropna(how="all")
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.15]})
for desp, sexo, c, lab in [(-0.2, "HOMBRE", PALETA[0], "Hombres"), (0.2, "MUJER", PALETA[1], "Mujeres")]:
    b = ax[0].bar([i + desp for i in range(len(GRUPOS_EDAD))], let_se.loc[sexo], 0.4, color=c, label=lab)
    ax[0].bar_label(b, fmt="%.0f", fontsize=8, padding=2)
ax[0].set_xticks(range(len(GRUPOS_EDAD)))
ax[0].set_xticklabels(GRUPOS_EDAD)
ax[0].set_xlabel("Grupo de edad (años)")
ax[0].set_ylabel("Letalidad entre víctimas (%)")
ax[0].set_title("Por sexo y edad")
ax[0].legend(frameon=False, loc="upper left")
sns.heatmap(let, annot=True, fmt=".0f", cmap=CMAP_SEC, ax=ax[1], cbar_kws={"label": "Letalidad entre víctimas (%)"})
ax[1].set_xlabel("Grupo de edad (años; celdas con n < 30 omitidas)")
ax[1].set_ylabel("Rol en el siniestro")
ax[1].set_yticklabels([t.get_text().capitalize() for t in ax[1].get_yticklabels()], rotation=0)
ax[1].set_title("Por rol y edad")
guardar_fig(fig, "p08_vulnerabilidad")

### Respuesta a la pregunta 8

**Resultados**

- **16 410 víctimas** (con edad el 65.2 %). Hombre : mujer = **2.8 : 1**.
- Letalidad: peatón **15.9 %**, conductor 13.8 %, pasajero 7.2 %; hombres 12.7 % y mujeres 7.7 %.
- Por edad: 0-11 años 9.4 %, 12-17 8.4 %, 18-29 11.5 %, 30-64 13.9 %, **65 o más 19.7 %**.
- Por sexo y edad (hombres / mujeres): 0-11 años 7.5 / 11.7 %, 12-17 10.7 / 4.4 %, 18-29 12.7 / 7.5 %, 30-64 15.7 / 8.7 %, 65+ **21.6 / 16.3 %**. Solo en 0-11 años las mujeres mueren más (197 niñas y 239 niños víctimas).
- Peatón por edad: 9, 6, 14, 21 y **27 %** (0-11 a 65+); la del conductor se mantiene en 13 a 16 %.
- Edad mediana: fallecidos 33 años, lesionados 30.

**Análisis**

1. **Las víctimas son sobre todo hombres adultos,** y desde los 12 años ellos mueren más que las mujeres en todos los grupos de edad.
2. **El adulto mayor es el más vulnerable,** pero es un fenómeno de peatones: la letalidad del conductor casi no cambia con la edad.

**Limitaciones**

- La edad falta en el 35 % de las víctimas y las que no la tienen son menos letales (8.5 % frente a 12.9 %), así que los niveles por edad están algo sesgados.
- No se mide exposición. Ciclista y conductor ausente (10 víctimas cada uno) no se interpretan.

---
## Pregunta 9 · Qué y a quién: A quién afecta cada tipo de siniestro y vehículo

> **¿Qué tipos de siniestro y qué vehículos afectan a quién? Es decir, qué rol (conductor, pasajero o peatón), sexo y edad tienen las víctimas según el tipo de siniestro y el vehículo involucrado.**

**Técnicas:** tabla participantes x siniestros (join por sid), composición porcentual del perfil de las víctimas por grupo, fila de referencia con el total y heatmaps comparables lado a lado.

**Qué hace el código**

- Toma las 16 410 víctimas (lesionados + fallecidos) y las une con el siniestro por `sid`.
- Para cada tipo de siniestro (100 o más víctimas) y cada vehículo (los 6 más frecuentes) calcula el % de víctimas que son conductor, pasajero o peatón, hombre, menor de 18 años o de 65 años o más. Cada % se calcula sobre las personas con ese dato conocido. Agrega una fila con el total como referencia.
- Dibuja dos heatmaps con la misma escala: filas = tipo de siniestro (ordenado por % de peatones) o vehículo (ordenado por % de conductores); columnas = quién.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from limpieza_datos import cargar_datos
from utilidades import CMAP_SEC, guardar_fig, guardar_tabla, mostrar, seccion

sin, part = cargar_datos()
vic = part[part["condicion"].isin(["LESIONADO", "FALLECIDO"])].copy()                  # víctimas
vic["rol3"] = vic["rol"].map({"CONDUCTOR PRESENTE": "Conductor", "PASAJERO": "Pasajero", "PEATÓN": "Peatón"})
VEH = {"Motocicleta": "n_motocicleta", "Automóvil": "n_automovil", "Camioneta": "n_camioneta",
       "SUV": "n_suv", "Camión": "n_camion", "Bus": "n_bus"}
vic = vic.merge(sin[["sid"] + list(VEH.values())], on="sid", how="left")
COLS = ["Conductor", "Pasajero", "Peatón", "Hombres", "Menores de 18", "65 años o más"]


def perfil(d):
    """% de víctimas por rol, sexo y edad (cada % se calcula sobre las personas con ese dato conocido)."""
    r = d["rol3"].dropna()
    s = d["sexo"][d["sexo"].isin(["HOMBRE", "MUJER"])]
    e = d["grupo_edad"].dropna()
    return pd.Series({"Conductor": 100 * (r == "Conductor").mean(), "Pasajero": 100 * (r == "Pasajero").mean(),
                      "Peatón": 100 * (r == "Peatón").mean(), "Hombres": 100 * (s == "HOMBRE").mean(),
                      "Menores de 18": 100 * e.isin(["0-11", "12-17"]).mean(), "65 años o más": 100 * (e == "65+").mean(),
                      "víctimas": len(d)})


# 1) Perfil de las víctimas según el tipo de siniestro
seccion("1) A quién afecta cada tipo de siniestro (% de las víctimas)")
base = perfil(vic).rename("Todos los siniestros")
tipo = vic.groupby("tipo_siniestro").apply(perfil, include_groups=False)
tipo = tipo[tipo["víctimas"] >= 100].sort_values("Peatón", ascending=False)
tipo.index = [t.capitalize() for t in tipo.index]
mostrar(pd.concat([base.to_frame().T, tipo]).round(1), 20)
guardar_tabla(pd.concat([base.to_frame().T, tipo]), "p09_perfil_por_tipo_siniestro")

# 2) Perfil de las víctimas según el vehículo involucrado en el siniestro
seccion("2) A quién afecta cada vehículo (víctimas de siniestros en que participa ese vehículo)")
veh = pd.DataFrame({k: perfil(vic[vic[c] > 0]) for k, c in VEH.items()}).T
veh = veh[veh["víctimas"] >= 100].sort_values("Conductor", ascending=False)
mostrar(pd.concat([base.to_frame().T, veh]).round(1), 20)
guardar_tabla(pd.concat([base.to_frame().T, veh]), "p09_perfil_por_vehiculo")
fall = vic[vic["condicion"] == "FALLECIDO"]
f = fall.groupby("tipo_siniestro").apply(perfil, include_groups=False)
print("\nSolo fallecidos en atropellos y arrollamientos (% peatón):")
print(f.loc[["ATROPELLOS", "ARROLLAMIENTOS"], ["Peatón", "víctimas"]].round(1).to_string())
print("Solo fallecidos en siniestros con moto (% conductor):", round(perfil(fall[fall["n_motocicleta"] > 0])["Conductor"], 1))

# 3) Heatmaps comparables: filas = tipo de siniestro / vehículo, columnas = quién; primera fila = total
fig, ax = plt.subplots(1, 2, figsize=(17, 6.8), gridspec_kw={"width_ratios": [1, 1], "wspace": 0.45})
for a, tabla, titulo in [(ax[0], tipo, "Según el tipo de siniestro"), (ax[1], veh, "Según el vehículo involucrado")]:
    m = pd.concat([base.to_frame().T, tabla])[COLS]
    etiquetas = [f"{i} ({int(n):,})" for i, n in zip(m.index, pd.concat([base.to_frame().T, tabla])["víctimas"])]
    sns.heatmap(m, annot=True, fmt=".0f", cmap=CMAP_SEC, vmin=0, vmax=100, ax=a, cbar=False, linewidths=0.6, linecolor="white")
    a.set_yticklabels(etiquetas, rotation=0)
    a.set_xticklabels(COLS, rotation=30, ha="right")
    a.axhline(1, color="white", lw=5)                       # separa la referencia (todos los siniestros)
    a.axvline(3, color="white", lw=5)                       # separa rol | sexo y edad
    a.set_title(titulo)
    a.set_ylabel("")
    a.set_xlabel("% de las víctimas (entre paréntesis, nº de víctimas)")
guardar_fig(fig, "p09_a_quien_afecta")

### Respuesta a la pregunta 9

**Resultados**

| Grupo | Víctimas | Peatón | Conductor | Pasajero | Hombres | 65+ |
|---|---|---|---|---|---|---|
| **Todos los siniestros** | 16 410 | 14 % | 44 % | 41 % | 73 % | 6 % |
| Atropellos | 2 209 | **84 %** | 12 % | 4 % | 68 % | **19 %** |
| Arrollamientos | 149 | 69 % | 20 % | 11 % | 73 % | 15 % |
| Caída de pasajero | 464 | 2 % | 2 % | **97 %** | 70 % | 9 % |
| Con moto | 6 927 | 6 % | **61 %** | 33 % | **80 %** | 3 % |
| Con bus | 1 244 | 9 % | 26 % | **65 %** | 62 % | **10 %** |

- En los choques entre vehículos (frontal, lateral, posterior, estrellamiento) conductores y pasajeros se reparten casi por mitades y los peatones son menos de 5 %.
- Entre los fallecidos, el 88 % de los de atropellos y el 66 % de los de arrollamientos son peatones; en siniestros con moto, el 76 % de los fallecidos son conductores.

**Análisis**

1. **Cada tipo de siniestro afecta a alguien distinto:** los atropellos y arrollamientos, a peatones (con más personas mayores de lo habitual); la caída de pasajero, a pasajeros; los choques entre vehículos, a quienes van dentro.
2. **Cada vehículo tiene su perfil:** las motos afectan sobre todo a sus conductores y a hombres jóvenes; los buses, a pasajeros, con más mujeres y más personas de 65 años o más que el promedio.
3. Complementa la pregunta 8: allí se ve quién es más vulnerable (letalidad); aquí, qué siniestro y qué vehículo se asocia a cada persona.

**Limitaciones**

- Las víctimas de «siniestros con moto» o «con bus» son todas las del siniestro, no solo quienes iban en ese vehículo (puede haber peatones u ocupantes de otro vehículo). Un siniestro con moto y bus cuenta en ambos grupos.
- El sexo se conoce en 99 % de las víctimas y la edad en 65 %; los porcentajes de edad se calculan solo con quienes la tienen.
- Los menores de 18 son 9 % de las víctimas y los de 65 o más 6 %; sus diferencias entre filas son pequeñas y los grupos con pocas víctimas (arrollamientos, 149) se interpretan con cautela.
- Describe a quién afecta cada siniestro, no cuán probable es que le ocurra a cada grupo: no se mide exposición.

---
## Pregunta 10 · Consecuencias: Siniestros de alta consecuencia

> **¿Qué características tienen los siniestros de alta consecuencia (3 o más víctimas), cuánto concentran de las víctimas y se explican por la cantidad de personas expuestas?**

**Técnicas:** concentración de víctimas y fallecidos, comparación del % de alta consecuencia con y sin cada característica, personas involucradas por siniestro con y sin cada característica (exposición), barras agrupadas con filas compartidas y top 10 de siniestros.

**Qué hace el código**

- Define alta consecuencia como `n_victimas >= 3` y mide cuánto concentran esos siniestros de las víctimas y fallecidos.
- Para 14 características (bus, camión, moto, 3 o más vehículos, zona rural, noche, causa, tipo de siniestro, etc.) compara el % de alta consecuencia *con* y *sin* ella ("veces más frecuente") y el promedio de personas registradas por siniestro con y sin ella. Lista el top 10 de siniestros.
- Dibuja una figura de tres paneles que comparten el orden de filas: qué parte del total son los siniestros de alta consecuencia, el % con y sin cada característica y las personas por siniestro con y sin ella.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from limpieza_datos import cargar_datos
from utilidades import GRIS, PALETA, guardar_fig, guardar_tabla, mostrar, seccion

sin, part = cargar_datos()
UMBRAL = 3
sin["alta"] = sin["n_victimas"] >= UMBRAL
sin["n_personas"] = sin["sid"].map(part.groupby("sid").size()).fillna(0)       # personas registradas (víctimas + ilesos)

# 1) Concentración
seccion("1) Concentración de víctimas")
v = sin["n_victimas"]
pct_sin = 100 * sin["alta"].mean()
pct_vic = 100 * v[sin["alta"]].sum() / v.sum()
pct_fal = 100 * sin.loc[sin["alta"], "fallecidos"].sum() / sin["fallecidos"].sum()
print(f"Promedio de víctimas por siniestro: {v.mean():.2f} | máximo: {v.max()}")
print(f"Siniestros con {UMBRAL} o más víctimas: {int(sin['alta'].sum()):,} ({pct_sin:.1f}%) "
      f"-> concentran {pct_vic:.1f}% de las víctimas y {pct_fal:.1f}% de los fallecidos")
print(f"Personas por siniestro: alta consecuencia {sin.loc[sin['alta'], 'n_personas'].mean():.2f} vs resto {sin.loc[~sin['alta'], 'n_personas'].mean():.2f}")

# 2) Características: % de alta consecuencia y personas por siniestro, con y sin la característica
seccion("2) Características asociadas a la alta consecuencia")
car = {
    "Choque frontal": sin["tipo_siniestro"] == "CHOQUE FRONTAL", "3 o más vehículos": sin["n_vehiculos"] >= 3,
    "Participa un bus": sin["n_bus"] > 0, "Transporte público": sin["transporte_publico"], "Red vial estatal": sin["via_estatal"],
    "Zona rural": sin["zona"] == "RURAL", "Pérdida de pista": sin["tipo_siniestro"] == "PÉRDIDA DE PISTA",
    "Participa un camión": sin["n_camion"] > 0, "Fin de semana": sin["fin_semana"], "Feriado": sin["feriado"],
    "Causa: exceso de velocidad": sin["causa_grupo"] == "Exceso de velocidad", "Horario nocturno (18-05 h)": sin["nocturno"],
    "Participa una motocicleta": sin["involucra_moto"], "Causa: alcohol / sustancias": sin["causa_grupo"] == "Alcohol / sustancias",
}
filas = []
for k, m in car.items():
    con, sn = 100 * sin.loc[m, "alta"].mean(), 100 * sin.loc[~m, "alta"].mean()
    filas.append({"característica": k, "siniestros con ella": int(m.sum()), "% alta consecuencia (con)": con,
                  "% alta consecuencia (sin)": sn, "veces más frecuente": con / sn,
                  "personas por siniestro (con)": sin.loc[m, "n_personas"].mean(),
                  "personas por siniestro (sin)": sin.loc[~m, "n_personas"].mean()})
res = pd.DataFrame(filas).set_index("característica").sort_values("veces más frecuente", ascending=False)
mostrar(res.round(2), 20)
guardar_tabla(res, "p10_caracteristicas_alta_consecuencia")

# 3) Los 10 siniestros con más víctimas
seccion("3) Los 10 siniestros con más víctimas")
t10 = sin.nlargest(10, "n_victimas")[["fecha", "canton", "provincia", "tipo_siniestro", "n_vehiculos", "lesionados", "fallecidos", "causa_grupo"]]
mostrar(t10)
guardar_tabla(t10, "p10_top10_siniestros", indice=False)

# 4) Gráfico: concentración | % de alta consecuencia con y sin | personas por siniestro con y sin
fig, ax = plt.subplots(1, 3, figsize=(18, 6.2), gridspec_kw={"width_ratios": [0.55, 1.25, 1.05]})
for i, (lab, p) in enumerate([("Siniestros", pct_sin), ("Víctimas", pct_vic), ("Fallecidos", pct_fal)]):
    ax[0].bar(i, p, color=PALETA[1], width=0.6)
    ax[0].bar(i, 100 - p, bottom=p, color=GRIS, width=0.6)
    ax[0].text(i, p / 2 if p > 12 else p + 3, f"{p:.0f} %" if p >= 10 else f"{p:.1f} %", ha="center", va="center", fontsize=10,
               color="white" if p > 12 else "#242421", fontweight="bold")
ax[0].set_xticks(range(3))
ax[0].set_xticklabels(["Siniestros", "Víctimas", "Fallecidos"])
ax[0].set_ylabel("% del total")
ax[0].set_title(f"Qué parte son los de\n{UMBRAL}+ víctimas")
ax[0].text(1, 55, "Resto", ha="center", color="white", fontsize=9)
r = res.iloc[::-1]
y = range(len(r))
h = 0.38
b = ax[1].barh([i + h / 2 for i in y], r["% alta consecuencia (con)"], h, color=PALETA[0], label="CON la característica")
ax[1].barh([i - h / 2 for i in y], r["% alta consecuencia (sin)"], h, color=GRIS, label="SIN la característica")
for i, (_, f) in zip(y, r.iterrows()):
    ax[1].text(f["% alta consecuencia (con)"] + 0.2, i + h / 2, f"{f['veces más frecuente']:.2f}×", va="center", fontsize=8)
ax[1].set_yticks(list(y))
ax[1].set_yticklabels(r.index)
ax[1].axvline(pct_sin, color="#52514e", ls="--", lw=1)
ax[1].set_xlabel(f"% de siniestros con {UMBRAL} o más víctimas (× = veces más frecuente con que sin)")
ax[1].set_title("¿Qué los hace más frecuentes?")
ax[1].legend(frameon=False, loc="lower right")
ax[2].barh([i + h / 2 for i in y], r["personas por siniestro (con)"], h, color=PALETA[0])
ax[2].barh([i - h / 2 for i in y], r["personas por siniestro (sin)"], h, color=GRIS)
for i, (_, f) in zip(y, r.iterrows()):
    ax[2].text(f["personas por siniestro (con)"] + 0.04, i + h / 2, f"{f['personas por siniestro (con)']:.1f}", va="center", fontsize=8)
ax[2].set_yticks(list(y))
ax[2].set_yticklabels([])
ax[2].set_xlabel("Personas involucradas por siniestro (promedio)")
ax[2].set_title("¿Cuánta gente van expuesta?")
guardar_fig(fig, "p10_concentracion_y_exposicion")

### Respuesta a la pregunta 10

**Resultados**

- Promedio 0.95 víctimas por siniestro; máximo 41. **931 siniestros (5.4 %)** tienen 3 o más y concentran el **22.9 % de las víctimas** y el 16.6 % de los fallecidos.
- Veces más frecuente: choque frontal **2.41**, 3 o más vehículos **2.23**, bus **2.01**, transporte público 1.53, red estatal 1.44, zona rural 1.43. Velocidad 1.07, noche 1.01, moto 0.97, alcohol 0.82.
- Personas por siniestro con la característica: 3 o más vehículos **3.6**, bus **3.0**, choque frontal **2.6**, transporte público 2.5; moto 2.15 y alcohol 2.2, igual que el resto (2.15). Los de alta consecuencia tienen 4.85 personas en promedio contra 2.0 en los demás.
- El más grave: 41 lesionados en un estrellamiento en Santa Elena (19 de agosto).

**Análisis**

1. **Las víctimas están concentradas:** 1 de cada 20 siniestros explica casi 1 de cada 4 víctimas.
2. **Lo define cuánta gente va expuesta:** las tres características que más aumentan la frecuencia (choque frontal, varios vehículos, bus) son las que más personas reúnen por siniestro; moto y alcohol no aumentan ninguna de las dos. Moto y bus son más fatales (pregunta 7), pero solo el bus reúne a mucha gente.

**Limitaciones**

- El umbral de 3 víctimas es una decisión del equipo.
- Las razones comparan cada característica por separado, aunque varias se solapan (bus y transporte público, choque frontal y rural).
- La relación con las personas por siniestro no es universal: la pérdida de pista tiene más alta consecuencia (1.37 veces) con solo 1.6 personas por siniestro. Además, las víctimas forman parte de las personas registradas, así que los siniestros con muchas personas tienen más posibilidad de llegar a 3 víctimas por definición.

---
## Conclusiones

- **Cuándo.** Los siniestros se concentran en fin de semana, con el domingo como día más cargado, y entre las 17 y las 20 horas, con máximo a las 19. En el campo el patrón es más extremo que en la ciudad. Hubo 13 días críticos de 273, 11 de ellos entre junio y septiembre, y casi uno de cada cuatro fue feriado.
- **Dónde.** El volumen está en Guayas y Pichincha (más de la mitad de los siniestros), pero la gravedad destaca en provincias pequeñas como Esmeraldas, Zamora Chinchipe y Cañar. Las motos, los camiones y los buses pesan más en provincias distintas. La zona rural es el doble de grave que la urbana (14 % contra 7 % de siniestros con fallecidos) y la madrugada es la franja más letal.
- **Por qué.** El 94.5 % de los siniestros se asocia a factor humano, sobre todo distracción y exceso de velocidad, que además están entre las causas más graves.
- **Qué y a quién.** Los arrollamientos, las pérdidas de pista y los choques frontales son los tipos más graves, y los siniestros con bus o moto son más fatales que los demás. Las víctimas son sobre todo hombres adultos, pero los más vulnerables son los peatones de 65 años o más. Cada tipo de siniestro afecta a alguien distinto: los atropellos, a peatones; las motos, a sus conductores; los buses, a sus pasajeros.
- **Consecuencias.** El 5.4 % de los siniestros tiene tres o más víctimas, pero reúne el 23 % de todas. Los choques frontales, los siniestros con tres o más vehículos y los que involucran un bus son los más frecuentes entre ellos, y son también los que reúnen más personas por evento: la alta consecuencia depende de cuánta gente va expuesta.

**Límites.** El análisis es descriptivo y retrospectivo: caracteriza lo que ya ocurrió y no predice dónde ocurrirá lo próximo. Las comparaciones son asociaciones, no causas. La calidad del registro varía entre entes de control, no hay datos de exposición (parque automotor ni kilómetros recorridos) y la edad falta en una parte importante de las víctimas.